# Summary

# 1. Performing computations

In [ ]:
%config InlineBackend.figure_format = "retina"

import glob
import importlib
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

# Load configuration and find project root
if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = None
    for _p in [os.getcwd()] + sys.path:
        _curr = os.path.abspath(_p) if _p else ""
        while _curr and _curr != os.path.dirname(_curr):
            if os.path.exists(os.path.join(_curr, "scripts/analysis/analyze.py")):
                PROJECT_ROOT = _curr
                break
            _curr = os.path.dirname(_curr)
        if PROJECT_ROOT: break
    if not PROJECT_ROOT:
        PROJECT_ROOT = os.getcwd()

scripts_dir = os.path.abspath(os.path.join(PROJECT_ROOT, "scripts", "analysis"))
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

import display as _display_helpers
import utils

for _m in (utils, _display_helpers):
    importlib.reload(_m)

from utils import (
    CALLER_KEYS, CHROMHMM_DEFAULT, COMPOSITION_DISPLAY, COSINE, F1_DISPLAY,
    FULL, FULL_DISPLAY, FUNCTIONAL_TARGETS, FUNCTIONAL_TX_JACCARD, JACCARD,
    JACCARD_DISPLAY, JOINT_CHROMHMM, JOINT_HATCH, JOINT_KMEANS_HOMER,
    JOINT_KMEANS_MACS2, JOINT_KMEANS_OMNI, 
    BMM3_HOMER, BMM3_MACS2, BMM3_OMNI, JOINT_BMM3_HOMER, JOINT_BMM3_MACS2, JOINT_BMM3_OMNI,
    KAPPA, KAPPA_DISPLAY, KMEANS_HOMER,
    KMEANS_MACS2, KMEANS_OMNI, NOQH, NOQH_DISPLAY, POINT_SIZE
)
from display import header, show_all

FUNCTIONAL_TARGETS = tuple(t for t in FUNCTIONAL_TARGETS if t[0] != "functional_tx")


In [ ]:
COMPARISON_METRICS = [KAPPA]
COMPARISON_DOMAINS = [FULL]

COLLECTED_METRICS = COMPARISON_METRICS + [COSINE]

COSINE_DOMAINS = [NOQH]

CROSS_ASSAY_DOMAINS = [NOQH]

ENTROPY_DOMAINS = list(set(COMPARISON_DOMAINS))

COLLECTED_DOMAINS = list(set(COMPARISON_DOMAINS + COSINE_DOMAINS))


def metric_domains(metric):
    """The comparison domains `metric` is collected and ranked in."""
    return (COSINE_DOMAINS if utils.normalize_metric(metric) == COSINE
            else COMPARISON_DOMAINS)


METRICS_SLASH_LABEL = " / ".join(utils.metric_display(m) for m in COMPARISON_METRICS)
METRICS_LABEL = " and ".join(utils.metric_display(m) for m in COMPARISON_METRICS)
DOMAINS_LABEL = " and ".join(utils.domain_display(d) for d in COMPARISON_DOMAINS)
ENTROPY_DOMAINS_LABEL = " and ".join(utils.domain_display(d) for d in ENTROPY_DOMAINS)
COSINE_DOMAINS_LABEL = " and ".join(utils.domain_display(d) for d in COSINE_DOMAINS)
CROSS_ASSAY_DOMAINS_LABEL = " and ".join(utils.domain_display(d) for d in CROSS_ASSAY_DOMAINS)
print(f"  metrics {COMPARISON_METRICS} over {COMPARISON_DOMAINS}, "
      f"[{COSINE!r}] over {COSINE_DOMAINS}, "
      f"entropy over {ENTROPY_DOMAINS}")

OUT = "out/summary"
os.makedirs(OUT, exist_ok=True)


In [ ]:
METHODS = [
    # ChromHMM
    CHROMHMM_DEFAULT,
    JOINT_CHROMHMM,
    # HOMER
    KMEANS_HOMER,
    BMM3_HOMER,
    JOINT_KMEANS_HOMER,
    JOINT_BMM3_HOMER,
    # MACS2
    KMEANS_MACS2,
    BMM3_MACS2,
    JOINT_KMEANS_MACS2,
    JOINT_BMM3_MACS2,
    # OmniPeak
    KMEANS_OMNI,
    BMM3_OMNI,
    JOINT_KMEANS_OMNI,
    JOINT_BMM3_OMNI,
]

def is_joint(method):
    """True for a joint model: the family a method is scored and ranked in."""
    return str(method).startswith("joint")


INDIVIDUAL_METHODS = [m for m in METHODS if not is_joint(m)]
JOINT_METHODS = [m for m in METHODS if is_joint(m)]


DATASETS = ["encode", "sagaconf", "epi1000"]

DATASET_NAMES = {"encode": "ENCODE", "sagaconf": "SAGAconf", "epi1000": "1000 epigenomes"}

AXES = {
    "replicates": (
        "Replicate\nagreement", "high", METRICS_SLASH_LABEL,
        f"Agreement between segmentations of biological replicates of the same "
        f"condition, averaged over {METRICS_LABEL} and {DOMAINS_LABEL}."),
    "replicates_cosine": (
        "Replicate\ncomposition", "high", "Cosine",
        f"Agreement of state composition between biological replicates, "
        f"measured by Cosine similarity over {COSINE_DOMAINS_LABEL}."),
    "replicates_emissions": (
        "Replicate emission\nsimilarity", "high", "cosine of the state emissions",
        "Similarity of the state signatures a method learns from the two "
        "replicates of a sample: the mean cosine of the emission vectors of "
        "the matched states, over the one-to-one matching of the two state "
        "spaces. Higher means the same biology is described by the same state "
        "definitions."),
    "cross_sample": (
        "Cross-sample\nagreement", "high", METRICS_SLASH_LABEL,
        f"Agreement between segmentations of different samples, averaged over "
        f"{METRICS_LABEL} and {DOMAINS_LABEL}. The two families are compared "
        f"over different sample sets, which costs nothing because they are "
        f"ranked apart: on ENCODE the individual models are read over every "
        f"pair of the 15 datasets, the joint ones - which exist per dataset "
        f"rather than per sample - over the pairs of the three datasets that "
        f"have replicates."),
    "cross_sample_cosine": (
        "Cross-sample\ncomposition", "high", "Cosine",
        f"Agreement of state composition between different samples, "
        f"measured by Cosine similarity over {COSINE_DOMAINS_LABEL}."),
    "joint_indiv": (
        "Individual /\njoint agreement", "high", METRICS_SLASH_LABEL,
        f"Agreement of a sample segmented on its own with the same sample "
        f"segmented inside a joint model, averaged over {METRICS_LABEL} and "
        f"{DOMAINS_LABEL}."),
    "joint_indiv_cosine": (
        "Individual / joint\ncomposition agreement", "high", "Cosine",
        f"Agreement of state composition between individual and joint "
        f"segmentations, measured by Cosine similarity over {COSINE_DOMAINS_LABEL}."),
    "entropy": (
        "Transition Matrix\nEntropy stability", "low", "CV of entropy",
        "Coefficient of variation of the state transition matrix entropy "
        "across the samples of a dataset. Lower means the fragmentation "
        "is more consistent across samples. "
        f"Collected once per comparison domain ({ENTROPY_DOMAINS_LABEL}); each "
        f"domain is ranked only against itself."),
    "functional_atac": (
        "Active chromatin\nvs ATAC-seq (F1)", "high", F1_DISPLAY,
        f"Overlap of the active states (the promoter and enhancer families) "
        f"with the ATAC-seq peaks of the same sample, measured by {F1_DISPLAY}: "
        f"the harmonic mean of the annotated share of the state family and of "
        f"the share of the annotation the family covers."),
    "functional_tss": (
        "Tss states vs\nexpressed TSS (F1)", "high", F1_DISPLAY,
        f"Overlap of the Tss state family (Tss and its flanking states) with "
        f"the RNA-seq-expressed transcription start sites of the same sample, "
        f"+-2 kb, measured by {F1_DISPLAY}."),
    "functional_refseq_tss": (
        "Tss states vs\nRefSeq TSS (F1)", "high", F1_DISPLAY,
        f"Overlap of the Tss state family (Tss and its flanking states) with "
        f"the RefSeq transcription start sites, +-2 kb, measured by {F1_DISPLAY}."),
    "functional_tx_jaccard": (
        "Tx state vs\nexpressed genes (Jaccard)", "high", JACCARD_DISPLAY,
        f"Overlap of the Tx state alone (without TxWk) with the "
        f"RNA-seq-expressed gene bodies of the same sample, measured by "
        f"{JACCARD_DISPLAY} of the two bp sets - the quantity "
        f"analysis_encode.ipynb plots as \"{JACCARD_DISPLAY}: Tx state vs "
        f"expressed gene bodies\"."),
    "state_recovery": (
        "Recovery of requested\nstate count", "high", "fraction reaching k=15",
        "Fraction of segmentations that actually realize the 15 requested "
        "states instead of collapsing some of them."),
    "segments_stability": (
        "Segment-count\nstability", "low", "CV of segment number",
        "Coefficient of variation of the number of segments across the "
        "samples of a dataset — how much the granularity drifts with input."),
    "cross_assay_agreement": (
        "Cross-assay\nagreement", "high", METRICS_SLASH_LABEL,
        f"Agreement between the ChIP-seq and the Mint-ChIP segmentations of a "
        f"caller, over every ChIP x Mint pair of samples, so it asks how much "
        f"agreement survives a change of assay and of cell type together. "
        f"Read on raw states using remapping to align the two vocabularies, "
        f"over {METRICS_LABEL} and "
        f"{CROSS_ASSAY_DOMAINS_LABEL}. The callers are ranked inside each pair "
        f"and those ranks averaged, since a caller whose Mint-ChIP model "
        f"collapsed has fewer usable pairs than the others."),
    "cross_assay_agreement_cosine": (
        "Cross-assay\ncomposition agreement", "high", "Cosine",
        f"Agreement of state composition between the ChIP-seq and the "
        f"Mint-ChIP segmentations of a caller, on the same rematched raw state "
        f"labels and pairs as the axis above and ranked inside each pair the "
        f"same way, measured by Cosine similarity "
        f"over {CROSS_ASSAY_DOMAINS_LABEL}."),
    "cross_assay_segments_stability": (
        "Cross-assay\ncounts stability", "low", "CV of segment number",
        "Coefficient of variation of the number of segments between the "
        "ChIP-seq and the Mint-ChIP segmentation of a caller, over every "
        "ChIP x Mint pair of datasets - the pairs the two cross-assay "
        "agreement axes are read over. The callers are ranked inside each "
        "pair and those ranks averaged, since a caller whose Mint-ChIP model "
        "collapsed has fewer usable pairs than the others."),
}

AXIS_ORDER = [
    "state_recovery", "segments_stability",
    "entropy",
    "functional_atac", "functional_tss", "functional_refseq_tss",
    "functional_tx_jaccard",
    "cross_sample", "cross_sample_cosine",
    "replicates", "replicates_cosine",
    "replicates_emissions",
    "cross_assay_segments_stability", "cross_assay_agreement", "cross_assay_agreement_cosine",
    "joint_indiv", "joint_indiv_cosine",
]

AXIS_WEIGHTS = {"state_recovery": 5}

UNBOUNDED_AXES = {"entropy", "segments_stability",
                  "cross_assay_segments_stability"}

RANKING_GROUPS = {
    "Segmentation": ["state_recovery", "segments_stability"],
    "Biological assay": ["functional_tss", "functional_refseq_tss", "functional_tx_jaccard", "cross_sample"],
    "Cross-sample": ["cross_sample", "cross_sample_cosine"],
    "Replicates": ["replicates", "replicates_cosine", "replicates_emissions"],
    "Assay": ["cross_assay_segments_stability", "cross_assay_agreement", "cross_assay_agreement_cosine"],
    "Individual / Join": ["joint_indiv", "joint_indiv_cosine"]
}

VARIANT_MAIN_AXES = ["replicates", "replicates_cosine",
                     "cross_sample", "cross_sample_cosine",
                     "joint_indiv", "joint_indiv_cosine",
                     "cross_assay_agreement", "cross_assay_agreement_cosine",
                     "entropy"]


def axis_variants(main):
    """The per-domain and per-metric variants an axis is built from."""
    if main == "entropy":
        return [f"entropy_{mode}" for mode in ENTROPY_DOMAINS]

    cross_assay = main.startswith("cross_assay_agreement")

    if main.endswith("_cosine"):
        base = main[:-len("_cosine")]
        domains = CROSS_ASSAY_DOMAINS if cross_assay else COSINE_DOMAINS
        return [f"{base}_{COSINE}_{mode}" for mode in domains]

    domains = CROSS_ASSAY_DOMAINS if cross_assay else COMPARISON_DOMAINS
    return [f"{main}_{m}_{mode}"
            for m in COMPARISON_METRICS for mode in domains]


VARIANT_AXES = []
for main in VARIANT_MAIN_AXES:
    for v in axis_variants(main):
        if v in AXES:
            continue
        AXES[v] = (v, AXES[main][1], AXES[main][2], "Variant for ranking")
        VARIANT_AXES.append(v)

ALL_AXIS_CATEGORIES = AXIS_ORDER + VARIANT_AXES

EVIDENCE_COLUMNS = ["Axis", "Dataset", "Cell", "Method", "Value", "Raw",
                    "N", "Note"]


def canonical(name):
    return utils.normalize_method(name)


def axis_label(axis, oneline=False):
    label = AXES[str(axis)][0]
    return label.replace("\n", " ") if oneline else label


WORKDIRS, DATASET_KEYS = {}, {}
for dataset, config_file in [("encode", "config_encode.yaml"),
                             ("sagaconf", "config_sagaconf.yaml")]:
    with open(os.path.join(PROJECT_ROOT, config_file)) as f:
        dataset_config = yaml.safe_load(f)
    WORKDIRS[dataset] = dataset_config["workdir"]
    DATASET_KEYS[dataset] = list(dataset_config["datasets"])
WORKDIRS["epi1000"] = os.path.join(os.path.dirname(WORKDIRS["encode"]), "epi1000")

for dataset, workdir in WORKDIRS.items():
    present = os.path.isdir(os.path.join(os.path.expanduser(workdir), "out"))
    known = DATASET_KEYS.get(dataset)
    print(f"  {DATASET_NAMES[dataset]:16s} {workdir}"
          f"{f'   ({len(known)} datasets)' if known else ''}"
          f"{'' if present else '   (no out/ — run its notebook first)'}")


In [ ]:
pd.DataFrame([{"Axis": axis_label(a, oneline=True), "Metric": AXES[a][2],
               "Better": AXES[a][1], "What it measures": AXES[a][3]}
              for a in AXIS_ORDER])


## Reading the caches into one evidence table

In [ ]:
def _rows(axis, dataset, values, note, cell=None, n=None, raw=None):
    """Evidence rows for one axis, `values` keyed by method."""
    return [{"Axis": axis, "Dataset": dataset, "Cell": cell or dataset,
             "Method": method, "Value": float(value),
             "Raw": float((raw or {}).get(method, value)), "Note": note,
             "N": (n or {}).get(method, np.nan)}
            for method, value in values.items()
            if method in METHODS and value is not None and not pd.isna(value)]


_SAID = set()


def _say_once(message):
    """Print `message` unless this run has already printed it."""
    if message not in _SAID:
        _SAID.add(message)
        print(message)


def _common_units(df, keys, unit_col, where):
    """`df` cut down to the comparison units every method in it has a value
    for."""
    units = df[unit_col].astype(str)
    per_method = {method: set(group) for method, group in units.groupby(keys)}
    if not per_method:
        return df, keys
    common = set.intersection(*per_method.values())
    dropped = sorted(set.union(*per_method.values()) - common)
    if dropped:
        _say_once(f"  {where}: {len(dropped)} of {len(dropped) + len(common)} "
                  f"comparison units are missing for at least one method and "
                  f"go for all of them, so the averages cover the same units "
                  f"({', '.join(dropped)})")
    keep = units.isin(common)
    return df[keep], keys[keep]


def _unit_ranks(df, keys, value_col, unit_col, where, high=True):
    """Mean within-unit rank score per method, and the unit count behind it."""
    frame = pd.DataFrame({"method": list(keys),
                          "unit": list(df[unit_col].astype(str)),
                          "value": list(df[value_col].astype(float))})
    units = frame["unit"].nunique()
    scores, alone = {}, 0
    for _, side in frame.groupby(["unit", frame["method"].map(is_joint)]):
        if len(side) < 2:
            alone += 1
            continue
        ranks = side["value"].rank(ascending=not high, method="average")
        for method, rank in zip(side["method"], ranks):
            scores.setdefault(method, []).append(
                (len(side) - rank) / (len(side) - 1))
    reached = frame.groupby("method")["unit"].nunique()
    partial = reached[reached < units]
    if len(partial):
        _say_once(f"  {where}: "
                  + ", ".join(f"{m} reached {n} of {units} comparison units"
                              for m, n in partial.items())
                  + " - the methods are ranked inside each unit and those "
                    "ranks averaged, so a unit one method missed costs "
                    "neither that method the axis nor the others the unit")
    if alone:
        _say_once(f"  {where}: {alone} comparison unit(s) carry a single model "
                  f"of its family, so they hold no ranking and are left out")
    return ({m: float(np.mean(v)) for m, v in scores.items()},
            {m: len(v) for m, v in scores.items()})


def _aggregate(df, method_col, value_col, mapper=canonical, unit_col=None,
               where=None, balance="units", high=True):
    """Mean and count of `value_col` per method."""
    keys = df[method_col].map(mapper)
    keep = keys.notna() & df[value_col].notna()
    df, keys = df.loc[keep], keys[keep]
    if unit_col is None:
        grouped = df.groupby(keys)[value_col]
        return grouped.mean().to_dict(), grouped.size().to_dict()
    if balance == "ranks":
        return _unit_ranks(df, keys, value_col, unit_col, where, high=high)
    df, keys = _common_units(df, keys, unit_col, where)
    grouped = df.groupby(keys)[value_col]
    return grouped.mean().to_dict(), grouped.size().to_dict()


def _no_cache(where, what):
    print(f"  {where}: no {what}, that variant is omitted")


def _find_column(df, *candidates):
    """The first of `candidates` among the columns of `df`,
    case-insensitively."""
    lookup = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate is not None and str(candidate).lower() in lookup:
            return lookup[str(candidate).lower()]
    return None


def _domain_rows(df, mode, where):
    """The rows of a long cache belonging to comparison domain `mode`."""
    col = _find_column(df, "Mode")
    if col is None:
        _no_cache(where, "Mode column")
        return None
    modes = df[col].astype(str)
    rows = df[modes.map(utils.normalize_domain) == utils.normalize_domain(mode)]
    if rows.empty:
        _no_cache(where, f"{mode} rows (the cache has "
                         f"{sorted(modes.unique())})")
        return None
    return rows


def _metric_column(df, metric, where, mode=None):
    """The column holding `metric`; pass `mode` for one column per domain."""
    spellings = utils.metric_aliases(metric)
    if mode is None or utils.normalize_domain(mode) == FULL:
        candidates = list(spellings)
    else:
        candidates = [f"{spelling}_{mode}" for spelling in spellings]
    col = _find_column(df, *candidates)
    if col is None:
        _no_cache(where, f"a {metric} column (looked for {candidates})")
    return col


def _metric_rows(rows, axis, mode, dataset, where, note, method_col="Method",
                 unit_col="Dataset", balance="units", gate=True, wide=False):
    """One `{axis}_{metric}_{mode}` variant per metric collected in `mode`."""
    evidence = []
    for metric in COLLECTED_METRICS:
        if gate and mode not in metric_domains(metric):
            continue
        col = _metric_column(rows, metric, where, mode if wide else None)
        if col is None:
            continue
        values, counts = _aggregate(rows, method_col, col, unit_col=unit_col,
                                    where=where, balance=balance)
        raw = None
        if balance == "ranks":
            raw, _ = _aggregate(rows, method_col, col, unit_col=unit_col,
                                where=where, balance="units")
        evidence += _rows(f"{axis}_{metric}_{mode}", dataset, values,
                          f"{note} ({utils.metric_display(metric)}, "
                          f"{utils.domain_display(mode)})", n=counts, raw=raw)
    return evidence


def _long_cache_rows(df, axis, dataset, where, note, domains=None, **kwargs):
    """The evidence of a long cache that carries the domain in a Mode
    column."""
    evidence = []
    for mode in domains or COLLECTED_DOMAINS:
        rows = _domain_rows(df, mode, where)
        if rows is not None:
            evidence += _metric_rows(rows, axis, mode, dataset, where, note,
                                     **kwargs)
    return evidence


def _cv(values):
    values = np.asarray([v for v in values if v is not None and not pd.isna(v)],
                        dtype=float)
    if len(values) < 2 or values.mean() == 0:
        return np.nan
    return float(values.std(ddof=1) / values.mean())


def _segments_stability(dataset, df, method_col, segments_col, cell=None,
                        unit_col=None, where=None):
    keys = df[method_col].map(canonical)
    keep = keys.notna()
    df, keys = df.loc[keep], keys[keep]
    if unit_col is not None:
        df, keys = _common_units(df, keys, unit_col, where)
    grouped = df.groupby(keys)[segments_col]
    values = {method: _cv(group.values) for method, group in grouped}
    return _rows("segments_stability", dataset, values, cell=cell,
                 note=f"CV of the segment number over {len(df)} "
                      f"segmentations", n=grouped.size().to_dict())


def _entropy_stability(dataset, axis, df, method_col, value_col, unit_col=None,
                       cell=None, where=None, note=None):
    keys = df[method_col].map(canonical)
    keep = keys.notna()
    df, keys = df.loc[keep], keys[keep]
    if unit_col is not None:
        df, keys = _common_units(df, keys, unit_col, where)
    grouped = df.groupby(keys)[value_col]
    values = {method: _cv(group.values) for method, group in grouped}
    return _rows(axis, dataset, values, cell=cell,
                 note=f"{note} over {len(df)} segmentations",
                 n=grouped.size().to_dict())


def _replicate_emissions(out, datasets):
    """rep1-vs-rep2 emission similarity of every method, one row per
    dataset."""
    frames = []
    for ds in datasets:
        for rel in ("matched/comparison_all_pairs.tsv",
                    "matched/joint/comparison_all_pairs.tsv"):
            path = f"{out}/{ds}/{rel}"
            if not os.path.exists(path):
                continue
            df = _read(path)
            if df is None or utils.EMISSION not in df.columns:
                continue
            seg1, seg2 = df["seg1"].astype(str), df["seg2"].astype(str)
            pairs = [utils.is_replicate(a) and utils.should_compare(a, b)
                     for a, b in zip(seg1, seg2)]
            rows = df[pairs].dropna(subset=[utils.EMISSION])
            if rows.empty:
                continue
            frames.append(pd.DataFrame({
                "Dataset": ds,
                "Method": rows["seg1"].astype(str).str[:-len("_rep1")],
                utils.EMISSION: rows[utils.EMISSION].astype(float),
            }))
    if not frames:
        return None
    return (pd.concat(frames, ignore_index=True)
            .drop_duplicates(subset=["Dataset", "Method"]))


def _emission_rows(dataset, out, datasets, excluded, where):
    """The emission similarity evidence of one dataset, empty when uncached."""
    emissions = _drop_excluded(_replicate_emissions(out, datasets), excluded)
    if emissions is None or emissions.empty:
        _no_cache(where, f"{utils.EMISSION} in the all-pairs comparison "
                         f"tables")
        return []
    values, counts = _aggregate(emissions, "Method", utils.EMISSION,
                                unit_col="Dataset",
                                where=f"{where} all-pairs comparison tables")
    return _rows("replicates_emissions", dataset, values,
                 f"rep1 vs rep2 cosine of the matched state emission vectors, "
                 f"over {len(emissions)} segmentation pairs", n=counts)


def _read(path, **kwargs):
    if not os.path.exists(path):
        print(f"  missing {path}")
        return None
    if path.endswith(".pkl"):
        return pd.read_pickle(path)
    sep = "\t" if path.endswith(".tsv") else ","
    return pd.read_csv(path, sep=sep, **kwargs)


In [ ]:
ENCODE_DATASETS = DATASET_KEYS["encode"]
SAGACONF_DATASETS = DATASET_KEYS["sagaconf"]

def _scope(segmentation):
    return "replicates" if str(segmentation).endswith(("_rep1", "_rep2")) else "pooled"


def _encode_frames(out, filename, columns, datasets=ENCODE_DATASETS, label="encode"):
    frames = []
    for ds in datasets:
        for rel in (f"matched/{filename}", f"matched/joint/{filename}"):
            path = f"{out}/{ds}/{rel}"
            if not os.path.isdir(os.path.dirname(path)):
                continue
            df = _read(path)
            if df is None:
                continue
            df = df.copy()
            df["Method"] = (df["segmentation"].astype(str)
                            .str.replace(r"_rep\d$", "", regex=True))
            df["Dataset"] = ds
            df["Cell"] = f"{label}:" + df["segmentation"].map(_scope)
            df["Segmentation"] = ds + ":" + df["segmentation"].astype(str)
            frames.append(df[["Dataset", "Segmentation", "Method", "Cell"] + columns])
    if not frames:
        return None
    return (pd.concat(frames, ignore_index=True)
            .drop_duplicates(subset=["Segmentation"]))


N_STATES = 15

INCOMPLETE_COLUMNS = ["Dataset", "Segmentation", "Method", "n_states",
                      "Covered_bp", "Share", "Reason"]


def no_incomplete():
    """The empty unusable-segmentations frame, for a dataset with no gate."""
    return pd.DataFrame(columns=INCOMPLETE_COLUMNS)


def incomplete_segmentations(workdir, datasets=ENCODE_DATASETS, label="encode",
                             min_share=0.5, min_states=N_STATES):
    """The segmentations of `datasets` no comparison should be read off."""
    out = os.path.join(os.path.expanduser(workdir), "out")
    stats = _encode_frames(out, "segment_stats.tsv",
                          ["n_states", "n_segments", "mean_length"],
                          datasets=datasets, label=label)
    if stats is None:
        return no_incomplete()
    stats = stats.assign(Covered_bp=stats["n_segments"] * stats["mean_length"])
    median = stats.groupby("Dataset")["Covered_bp"].transform("median")
    stats["Share"] = stats["Covered_bp"] / median
    truncated = stats["Share"] < min_share
    collapsed = stats["n_states"] < min_states
    bad = stats[truncated | collapsed].copy()
    reasons = []
    for share, n_states in zip(bad["Share"], bad["n_states"]):
        why = []
        if share < min_share:
            why.append(f"covers {share:.2%} of the dataset median")
        if n_states < min_states:
            why.append(f"{int(n_states)} of {min_states} states")
        reasons.append(", ".join(why))
    bad["Reason"] = reasons
    bad["Method"] = bad["Method"].map(canonical)
    return bad[INCOMPLETE_COLUMNS]


def _excluded_pairs(incomplete, announce=True):
    reasons = dict(zip(zip(incomplete["Dataset"], incomplete["Method"]),
                       incomplete["Reason"]))
    pairs = {(ds, method)
             for ds, method in zip(incomplete["Dataset"], incomplete["Method"])
             if method is not None}
    if pairs and announce:
        print("  excluding unusable segmentations: "
              + ", ".join(f"{ds}/{m} ({reasons[(ds, m)]})"
                          for ds, m in sorted(pairs)))
    return pairs


def _excluded_comparisons(pairs):
    return {(ds, key) for ds, method in pairs
            for key in CALLER_KEYS.get(utils.caller_key(method), (method,))}


def _drop_excluded(df, excluded, method_col="Method", dataset_cols=("Dataset",)):
    """`df` without the rows that name an excluded (dataset, method) pair."""
    if not excluded or df is None or df.empty:
        return df
    methods = df[method_col].map(canonical)
    keep = np.ones(len(df), dtype=bool)
    for col in dataset_cols:
        sides = df[col].astype(str).str.split("/")
        keep &= np.array([not any((side, m) in excluded for side in row)
                          for row, m in zip(sides, methods)])
    return df[keep]


FUNCTIONAL_METRICS = {axis: F1_DISPLAY for axis, _, _, _ in FUNCTIONAL_TARGETS}
FUNCTIONAL_METRICS[FUNCTIONAL_TX_JACCARD[0]] = JACCARD_DISPLAY

FUNCTIONAL_LABELS = {axis: label for axis, label, _, _ in FUNCTIONAL_TARGETS}
FUNCTIONAL_LABELS[FUNCTIONAL_TX_JACCARD[0]] = FUNCTIONAL_TX_JACCARD[1]


def _encode_functional(out, label="encode"):
    """Every segmentation on every functional target, one row per
    annotation."""
    rows = []
    for pattern, scope in ((f"{out}/*/matched/*/report.tsv", "pooled"),
                           (f"{out}/*/rep*/matched/*/report.tsv", "replicates")):
        for path in sorted(glob.glob(pattern)):
            dirpath = os.path.dirname(path)
            ds = os.path.relpath(dirpath, out).split(os.sep)[0]
            method = canonical(os.path.basename(dirpath))
            if method is None:
                continue
            found = [(axis, utils.annotation_f1(dirpath, states, prefix))
                     for axis, _, states, prefix in FUNCTIONAL_TARGETS]
            axis, _, state, prefix = FUNCTIONAL_TX_JACCARD
            found.append((axis, utils.annotation_jaccard(dirpath, state, prefix)))
            for axis, found_rows in found:
                for row in found_rows:
                    rows.append({"Dataset": ds, "Method": method, "Axis": axis,
                                 "Cell": f"{label}:{scope}",
                                 "Label": row["Label"],
                                 "Value": float(row[FUNCTIONAL_METRICS[axis]])})
    return pd.DataFrame(rows) if rows else None


def collect_encode(workdir):
    out = os.path.join(os.path.expanduser(workdir), "out")
    print(f"ENCODE caches in {out}")
    evidence = []
    incomplete = incomplete_segmentations(workdir)
    excluded = _excluded_pairs(incomplete)

    rep = _read(f"{out}/df_joint_rep.pkl")
    if rep is not None:
        evidence += _long_cache_rows(
            _drop_excluded(rep, excluded), "replicates", "encode",
            "ENCODE df_joint_rep.pkl", "rep1 vs rep2")

    evidence += _emission_rows("encode", out, ENCODE_DATASETS, excluded,
                               "ENCODE")

    pairs = _read(f"{out}/comparison_table.tsv")
    if pairs is not None:
        mint_a = pairs["ds_a"].astype(str).str.endswith("_mint")
        mint_b = pairs["ds_b"].astype(str).str.endswith("_mint")
        same_assay = mint_a == mint_b
        if not same_assay.all():
            print(f"  ENCODE comparison_table.tsv: {int((~same_assay).sum())} of "
                  f"{len(pairs)} rows are ChIP x Mint-ChIP pairs, left to the "
                  f"cross-assay axes")
        pairs = pairs[same_assay]
        pairs = _drop_excluded(pairs, excluded, method_col="method",
                               dataset_cols=("ds_a", "ds_b"))
        pairs = pairs.assign(Pair=pairs["ds_a"].astype(str) + "/"
                                  + pairs["ds_b"].astype(str))
        for mode in COLLECTED_DOMAINS:
            evidence += _metric_rows(
                pairs, "cross_sample", mode, "encode",
                "ENCODE comparison_table.tsv", "all same-assay pairs",
                method_col="method", unit_col="Pair", wide=True)

    cross_sample = _drop_excluded(_read(f"{out}/df_cross_sample.pkl"), excluded)
    if cross_sample is None:
        _no_cache("ENCODE", "df_cross_sample.pkl (run the cross-sample section "
                            "of analysis_encode.ipynb)")
    else:
        joint_rows = cross_sample[
            cross_sample["SameAssay"].fillna(True).astype(bool)
            & cross_sample["Method"].map(canonical).map(is_joint)]
        if joint_rows.empty:
            _no_cache("ENCODE", "joint models in df_cross_sample.pkl")
        else:
            evidence += _long_cache_rows(
                joint_rows, "cross_sample", "encode",
                "ENCODE df_cross_sample.pkl",
                "mean rank among the joint models of each same-assay sample "
                "pair, on rep1 with the state spaces realigned; Raw is the "
                "metric itself, over the pairs every joint model reached",
                balance="ranks")

    cross_assay = _drop_excluded(_read(f"{out}/df_cross_assay.pkl"), excluded)
    if cross_assay is None:
        _no_cache("ENCODE", "df_cross_assay.pkl (run the cross-assay section "
                            "of analysis_encode.ipynb)")
    else:
        evidence += _long_cache_rows(
            cross_assay, "cross_assay_agreement", "encode",
            "ENCODE df_cross_assay.pkl",
            "mean rank among the callers of each ChIP vs Mint-ChIP pair, on "
            "rematched raw states; Raw is the metric itself, over the pairs "
            "every caller reached",
            domains=CROSS_ASSAY_DOMAINS, gate=False, balance="ranks")

    joint = _read(f"{out}/df_joint_indiv.pkl")
    if joint is not None:
        evidence += _long_cache_rows(
            _drop_excluded(joint, _excluded_comparisons(excluded)),
            "joint_indiv", "encode", "ENCODE df_joint_indiv.pkl",
            "individual vs joint")

    for mode in ENTROPY_DOMAINS:
        suffix = "" if mode == FULL else f"_{mode}"
        entropy = _drop_excluded(
            _encode_frames(out, f"entropy_summary{suffix}.tsv", ["total_entropy"]),
            excluded)
        if entropy is None:
            _no_cache("ENCODE", f"entropy_summary{suffix}.tsv "
                                f"({utils.domain_display(mode)} transition matrix entropy)")
            continue
        for cell, group in entropy.groupby("Cell"):
            evidence += _entropy_stability(
                "encode", f"entropy_{mode}", group, "Method", "total_entropy",
                unit_col="Dataset", cell=cell,
                where=f"ENCODE entropy_summary{suffix}.tsv ({cell})",
                note=f"CV of {utils.domain_display(mode)} transition matrix entropy of the "
                     f"{cell} segmentations")

    functional = _drop_excluded(_encode_functional(out), excluded)
    if functional is not None:
        for (axis, cell), group in functional.groupby(["Axis", "Cell"]):
            where = f"ENCODE enrichment ({axis}, {cell})"
            values, counts = _aggregate(group, "Method", "Value",
                                        mapper=lambda m: m,
                                        unit_col="Dataset", where=where)
            datasets = ", ".join(sorted(set.intersection(*(
                set(rows["Dataset"]) for _, rows in group.groupby("Method")))))
            evidence += _rows(axis, "encode", values,
                              f"{FUNCTIONAL_METRICS[axis]}, "
                              f"{FUNCTIONAL_LABELS[axis]}, over the {cell} "
                              f"segmentations of {datasets}",
                              cell=cell, n=counts)

    all_segments = _encode_frames(out, "segment_stats.tsv",
                                  ["n_states", "n_segments"])
    if all_segments is not None:
        # Before the gate: this axis measures the collapse the gate excludes for.
        fidelity = all_segments.assign(
            ok=all_segments["n_states"].eq(N_STATES).astype(float))
        for cell, group in fidelity.groupby("Cell"):
            values, counts = _aggregate(
                group, "Method", "ok", unit_col="Dataset",
                where=f"ENCODE segment_stats.tsv fidelity ({cell})")
            evidence += _rows("state_recovery", "encode", values,
                              f"fraction of the {cell} segmentations realizing "
                              f"{N_STATES} states", cell=cell, n=counts)

    segments = _drop_excluded(all_segments, excluded)
    if segments is not None:
        for cell, group in segments.groupby("Cell"):
            evidence += _segments_stability(
                "encode", group, "Method", "n_segments", cell=cell,
                unit_col="Dataset",
                where=f"ENCODE segment_stats.tsv stability ({cell})")
            chip_dss = [ds for ds in ENCODE_DATASETS if not ds.endswith("_mint")]
            mint_dss = [ds for ds in ENCODE_DATASETS if ds.endswith("_mint")]
            counts_by = (group[group["Method"].isin(METHODS)]
                         .groupby(["Method", "Dataset"])["n_segments"].mean())
            callers = [m for m in counts_by.index.get_level_values(0).unique()
                       if not is_joint(m)]
            cross = pd.DataFrame(
                [{"Method": m, "Pair": f"{chip}/{mint}",
                  "CV": _cv([counts_by[(m, chip)], counts_by[(m, mint)]])}
                 for m in callers for chip in chip_dss for mint in mint_dss
                 if (m, chip) in counts_by.index
                 and (m, mint) in counts_by.index],
                columns=["Method", "Pair", "CV"]).dropna(subset=["CV"])
            where = f"ENCODE cross-assay counts stability ({cell})"
            if cross.empty:
                _say_once(f"  {where}: no ChIP x Mint-ChIP dataset pair has a "
                          f"segmentation on both sides in this scope, so the "
                          f"axis is left to the other scope")
            else:
                res_values, res_counts = _aggregate(
                    cross, "Method", "CV", unit_col="Pair", where=where,
                    balance="ranks")
                res_raw, _ = _aggregate(cross, "Method", "CV", unit_col="Pair",
                                        where=where, balance="units")
                evidence += _rows(
                    "cross_assay_segments_stability", "encode", res_values,
                    f"mean rank among the callers of each of the "
                    f"{cross['Pair'].nunique()} ChIP vs Mint-ChIP dataset "
                    f"pairs, by CV of the segment number between the two "
                    f"assays; Raw is the CV itself, over the pairs every "
                    f"caller reached", cell=cell, n=res_counts, raw=res_raw)
    return evidence, incomplete


In [ ]:
def collect_sagaconf(workdir):
    out = os.path.join(os.path.expanduser(workdir), "out")
    print(f"SAGAconf caches in {out}")
    evidence = []
    incomplete = incomplete_segmentations(workdir, datasets=SAGACONF_DATASETS,
                                          label="sagaconf")
    excluded = _excluded_pairs(incomplete)

    def rows_of(axis, caches, label, note, drop):
        """The evidence of a SAGAconf cache pair, one file per comparison
        domain."""
        collected = []
        for mode in COLLECTED_DOMAINS:
            cache = caches.get(mode)
            df = _read(f"{out}/{cache}") if cache else None
            if df is None:
                _no_cache("SAGAconf", f"{utils.domain_display(mode)} {label} "
                                      f"({cache or 'no cache for this domain'})")
                continue
            collected += _metric_rows(_drop_excluded(df, drop), axis, mode,
                                      "sagaconf", f"SAGAconf {cache}", note)
        return collected

    evidence += rows_of("replicates",
                        {FULL: "df_rep.pkl", NOQH: "df_rep_noqh.pkl"},
                        "replicate agreement", "rep1 vs rep2", excluded)

    evidence += _emission_rows("sagaconf", out, SAGACONF_DATASETS, excluded,
                               "SAGAconf")

    evidence += rows_of("cross_sample",
                        {FULL: "df_cross_sample.pkl",
                         NOQH: "df_cross_sample_noqh.pkl"},
                        "cross-sample agreement", "all cell-line pairs",
                        excluded)

    evidence += rows_of("joint_indiv",
                        {FULL: "df_ji.pkl", NOQH: "df_ji_noqh.pkl"},
                        "individual vs joint agreement", "individual vs joint",
                        _excluded_comparisons(excluded))

    ENTROPY_CACHES = {FULL: "df_entropy.pkl", NOQH: "df_entropy_noqh.pkl"}
    for mode in ENTROPY_DOMAINS:
        cache = ENTROPY_CACHES.get(mode)
        entropy = _read(f"{out}/{cache}") if cache else None
        if entropy is None:
            print(f"  no {utils.domain_display(mode)} transition matrix entropy for SAGAconf "
                  f"(analysis_sagaconf.ipynb runs compare with skip_noqh=True); "
                  f"the entropy_{mode} variant omits it")
            continue
        entropy = _drop_excluded(entropy, excluded)
        evidence += _entropy_stability(
            "sagaconf", f"entropy_{mode}", entropy, "Method", "total_entropy",
            unit_col="Dataset", where=f"SAGAconf {cache}",
            note=f"CV of {utils.domain_display(mode)} transition matrix entropy")

    segments = _read(f"{out}/df_segs.pkl")
    if segments is not None:
        fidelity = segments.assign(
            ok=segments["n_states"].eq(N_STATES).astype(float))
        values, counts = _aggregate(fidelity, "Method", "ok",
                                    unit_col="Dataset",
                                    where="SAGAconf df_segs.pkl fidelity")
        evidence += _rows("state_recovery", "sagaconf", values,
                          f"fraction of the 5 cell lines x 2 replicates realizing "
                          f"{N_STATES} states", n=counts)
        evidence += _segments_stability("sagaconf",
                                        _drop_excluded(segments, excluded),
                                        "Method", "n_segments",
                                        unit_col="Dataset",
                                        where="SAGAconf df_segs.pkl stability")
    return evidence, incomplete


EPI1000_REF_INDIV_LABELS = ("ref 15", "individual", "ref_15")


def _drop_reference_indiv(df, where, col="Method"):
    """`df` without the rows of the published individual 15-state model."""
    if df is None:
        return None
    is_ref = df[col].astype(str).str.strip().str.lower().isin(EPI1000_REF_INDIV_LABELS)
    if is_ref.any():
        _say_once(f"  {where}: {int(is_ref.sum())} published individual "
                  f"15-state rows dropped, they are not the de-novo "
                  f"{CHROMHMM_DEFAULT}")
    return df[~is_ref]


def _reference_joint_only(df, where, col="Method"):
    """The `joint_chromhmm` rows of a cache holding only the reference
    models."""
    if df is None:
        return None
    keep = df[col].map(canonical) == JOINT_CHROMHMM
    if not keep.all():
        _say_once(f"  {where}: {int((~keep).sum())} published individual "
                  f"15-state rows dropped, {int(keep.sum())} joint ones kept "
                  f"as {JOINT_CHROMHMM}")
    return df[keep]


def _cross_sample_epi1000(out):
    by_method = {}
    for path in sorted(glob.glob(f"{out}/pw_cache_*.pkl")):
        cache = pd.read_pickle(path)
        df = cache.get("df") if isinstance(cache, dict) else None
        if df is None or df.empty:
            continue
        key = canonical(df["Method"].iloc[0])
        if key is None:
            continue
        previous = by_method.get(key)
        if previous is None or os.path.getmtime(path) > os.path.getmtime(previous[0]):
            by_method[key] = (path, df)
        if previous is not None:
            ignored = min([previous[0], path], key=os.path.getmtime)
            print(f"  1000 epigenomes: {os.path.basename(ignored)} is an older "
                  f"copy of the {key} pairs, ignored")
    frames = [df for _, df in by_method.values()]

    df_15 = _reference_joint_only(_read(f"{out}/df_pw_15.csv"),
                                  "1000 epigenomes df_pw_15.csv")
    if df_15 is not None and not df_15.empty:
        frames.append(df_15)

    if not frames:
        print(f"  missing {out}/pw_cache_*.pkl")
        return None
    return pd.concat(frames, ignore_index=True)


def collect_epi1000(workdir):
    out = os.path.join(os.path.expanduser(workdir), "out")
    print(f"1000-epigenomes caches in {out}")
    evidence = []

    rep = _drop_reference_indiv(_read(f"{out}/df_replicates.csv"),
                                "1000 epigenomes df_replicates.csv")
    if rep is not None:
        rep = rep.assign(Pair=rep["EID1"].astype(str) + "/"
                              + rep["EID2"].astype(str))
        evidence += _long_cache_rows(rep, "replicates", "epi1000",
                                     "1000 epigenomes df_replicates.csv",
                                     "rep1 vs rep2", unit_col="Pair")

    pairs = _cross_sample_epi1000(out)
    if pairs is not None:
        print("  1000 epigenomes: the cross-sample caches carry no pair "
              "identity, so the sampled pairs behind each method's mean are "
              "not checked to be the same ones")
        evidence += _long_cache_rows(pairs, "cross_sample", "epi1000",
                                     "1000 epigenomes pw_cache_*.pkl",
                                     "sampled pairs", unit_col=None)

    joint = _read(f"{out}/df_joint_indiv_comparison.csv")
    if joint is not None:
        evidence += _long_cache_rows(
            joint, "joint_indiv", "epi1000",
            "1000 epigenomes df_joint_indiv_comparison.csv",
            "individual vs joint")

    emissions = _drop_reference_indiv(
        _read(f"{out}/df_replicates_emissions.csv"),
        "1000 epigenomes df_replicates_emissions.csv")
    if emissions is None:
        _no_cache("1000 epigenomes", "df_replicates_emissions.csv (run the "
                                     "replicate emission similarity cell)")
    else:
        col = _find_column(emissions, utils.EMISSION, utils.EMISSION_DISPLAY)
        if col is None:
            _no_cache("1000 epigenomes", f"a {utils.EMISSION} column")
        else:
            emissions = emissions.assign(
                Pair=emissions["EID1"].astype(str) + "/"
                     + emissions["EID2"].astype(str))
            values, counts = _aggregate(
                emissions, "Method", col, unit_col="Pair",
                where="1000 epigenomes df_replicates_emissions.csv")
            evidence += _rows("replicates_emissions", "epi1000", values,
                              f"rep1 vs rep2 cosine of the matched state "
                              f"emission vectors, over {len(emissions)} "
                              f"replicate pairs", n=counts)

    results = _read(f"{out}/df_results.csv")
    entropy_15 = _reference_joint_only(_read(f"{out}/df_entropy_15.csv"),
                                       "1000 epigenomes df_entropy_15.csv")
    segments_15 = _reference_joint_only(_read(f"{out}/df_segments_15.csv"),
                                        "1000 epigenomes df_segments_15.csv")
    if entropy_15 is not None and segments_15 is not None:
        wide_15 = (entropy_15.pivot_table(index=["Dataset", "Method"],
                                          columns="Mode", values="Entropy")
                   .rename(columns={FULL_DISPLAY: "Entropy", NOQH_DISPLAY: "Entropy_NOQH"})
                   .reset_index())
        res_15 = pd.merge(wide_15, segments_15, on=["Dataset", "Method"])
        res_15["N_States"] = 15
        if results is not None:
            results = pd.concat([results, res_15], ignore_index=True)
        else:
            results = res_15

    if results is not None:
        for mode in ENTROPY_DOMAINS:
            col = _metric_column(results, "Entropy",
                                 "1000 epigenomes df_results.csv", mode)
            if col is None:
                continue
            evidence += _entropy_stability(
                "epi1000", f"entropy_{mode}", results, "Method", col,
                unit_col="Dataset", where="1000 epigenomes df_results.csv entropy",
                note=f"CV of {utils.domain_display(mode)} transition matrix entropy")

        fidelity = results.assign(
            ok=results["N_States"].eq(N_STATES).astype(float))
        values, counts = _aggregate(
            fidelity, "Method", "ok", unit_col="Dataset",
            where="1000 epigenomes df_results.csv fidelity")
        evidence += _rows("state_recovery", "epi1000", values,
                          f"fraction of the 98 epigenomes realizing "
                          f"{N_STATES} states", n=counts)
        evidence += _segments_stability(
            "epi1000", results, "Method", "N_Segments", unit_col="Dataset",
            where="1000 epigenomes df_results.csv stability")

        short = results[results["N_States"] < N_STATES]
        if not short.empty:
            per_method = (short.groupby(short["Method"].map(canonical))
                          .size().sort_values(ascending=False))
            print(f"  1000 epigenomes: {len(short)} of {len(results)} "
                  f"segmentations realize fewer than {N_STATES} states "
                  f"({', '.join(f'{m} {n}' for m, n in per_method.items())}); "
                  f"they stay in the agreement axes, which have no gate for "
                  f"this dataset - see state_recovery for the size of it")
    return evidence, no_incomplete()


COLLECTORS = {"encode": collect_encode, "sagaconf": collect_sagaconf,
              "epi1000": collect_epi1000}


def build_evidence(workdirs):
    _SAID.clear()
    evidence = []
    incomplete = []
    for dataset, collect in COLLECTORS.items():
        if dataset in workdirs:
            rows, bad = collect(workdirs[dataset])
            evidence += rows
            incomplete.append(bad)

    df = pd.DataFrame(evidence, columns=EVIDENCE_COLUMNS)

    unknown = sorted(set(df["Axis"].astype(str)) - set(ALL_AXIS_CATEGORIES))
    if unknown:
        raise ValueError(
            f"collected evidence for axes no main axis is built from: "
            f"{unknown}. axis_variants() takes the agreement metrics from "
            f"COMPARISON_METRICS and the domains from COMPARISON_DOMAINS / "
            f"COSINE_DOMAINS / CROSS_ASSAY_DOMAINS - register the axis there "
            f"or stop collecting it.")

    df["Axis"] = pd.Categorical(df["Axis"], categories=ALL_AXIS_CATEGORIES,
                                ordered=True)
    df["Method"] = pd.Categorical(df["Method"], categories=METHODS, ordered=True)

    incomplete = [frame for frame in incomplete
                  if frame is not None and not frame.empty]
    df_incomplete = pd.concat(incomplete, ignore_index=True) if incomplete \
        else no_incomplete()

    return (df.sort_values(["Axis", "Dataset", "Cell", "Method"])
            .reset_index(drop=True), df_incomplete)


REFERENCE = "reference"

SEGMENT_COUNT_MODELS = [REFERENCE] + METHODS

SEGMENT_COUNT_COLUMNS = ["Dataset", "Cell", "Scope", "Segmentation", "Model",
                         "N_Segments", "N_States"]


def _encode_reference_ids():
    """The ref_chromhmm accession of every ENCODE dataset that names one."""
    with open(os.path.join(PROJECT_ROOT, "config_encode.yaml")) as f:
        datasets = yaml.safe_load(f)["datasets"]
    return {ds: info["ref_chromhmm"] for ds, info in datasets.items()
            if info.get("ref_chromhmm")}


def _encode_segment_counts(workdir, excluded):
    """Segment and state counts of every ENCODE segmentation, reference
    included; both scopes, kept apart in Scope."""
    out = os.path.join(os.path.expanduser(workdir), "out")
    stats = _encode_frames(out, "segment_stats.tsv", ["n_states", "n_segments"])
    if stats is None:
        _no_cache("ENCODE", "segment_stats.tsv (segment counts)")
        return None
    stats = _drop_excluded(stats, excluded)
    references = _encode_reference_ids()
    is_reference = [method == f"{references.get(ds)}_chromhmm"
                    for ds, method in zip(stats["Dataset"], stats["Method"])]
    stats = stats.assign(Model=np.where(is_reference, REFERENCE,
                                        stats["Method"].map(canonical)))
    return (stats.rename(columns={"n_segments": "N_Segments",
                                  "n_states": "N_States"})
            .assign(Scope=stats["Cell"].astype(str).str.rsplit(":", n=1).str[-1]))


def _sagaconf_segment_counts(workdir, excluded):
    """Segment and state counts of the SAGAconf segmentations; it has no
    reference."""
    out = os.path.join(os.path.expanduser(workdir), "out")
    segments = _drop_excluded(_read(f"{out}/df_segs.pkl"), excluded)
    if segments is None:
        return None
    return (segments.assign(Model=segments["Method"].map(canonical),
                            Scope="",
                            Segmentation=segments["Dataset"].astype(str) + ":"
                                         + segments["segmentation"].astype(str))
            .rename(columns={"n_segments": "N_Segments",
                             "n_states": "N_States"}))


def _epi1000_segment_counts(workdir, excluded):
    """Segment and state counts of the 1000-epigenomes models, one row per
    epigenome."""
    out = os.path.join(os.path.expanduser(workdir), "out")
    frames = []
    results = _read(f"{out}/df_results.csv")
    if results is not None:
        frames.append(results.assign(Model=results["Method"].map(canonical)))
    # The published 15-state joint model is the joint ChromHMM of this
    # dataset, not a reference bar of its own.
    published = _read(f"{out}/df_segments_15.csv")
    if published is not None:
        published = published[published["Method"].map(canonical)
                              == JOINT_CHROMHMM]
        frames.append(published.assign(Model=JOINT_CHROMHMM))
    if not frames:
        return None
    counts = pd.concat(frames, ignore_index=True)
    return counts.assign(Scope="",
                         Segmentation=counts["Dataset"].astype(str) + ":"
                                      + counts["Model"].astype(str))


SEGMENT_COUNT_COLLECTORS = {"encode": _encode_segment_counts,
                            "sagaconf": _sagaconf_segment_counts,
                            "epi1000": _epi1000_segment_counts}


def collect_segment_counts(workdirs, incomplete):
    """One row per segmentation of every dataset: its model and its count."""
    excluded = _excluded_pairs(incomplete, announce=False)
    frames = []
    for dataset, collect in SEGMENT_COUNT_COLLECTORS.items():
        if dataset not in workdirs:
            continue
        counts = collect(workdirs[dataset], excluded)
        if counts is None or counts.empty:
            continue
        counts = counts[counts["Model"].isin(SEGMENT_COUNT_MODELS)]
        frames.append(counts.assign(Cell=counts["Dataset"].astype(str), Dataset=dataset)
                      .reindex(columns=SEGMENT_COUNT_COLUMNS))
    if not frames:
        print("  no segment counts collected")
        return pd.DataFrame(columns=SEGMENT_COUNT_COLUMNS)
    counts = pd.concat(frames, ignore_index=True)
    counts["N_Segments"] = counts["N_Segments"].astype(float)
    counts["N_States"] = pd.to_numeric(counts["N_States"], errors="coerce")
    unknown = counts["N_States"].isna()
    if unknown.any():
        models = sorted(set(counts.loc[unknown, "Model"].astype(str)))
        print(f"  no state count for {int(unknown.sum())} segmentations "
              f"({', '.join(models)}) - re-run their notebook to cache it")
    counts["Dataset"] = pd.Categorical(counts["Dataset"], categories=DATASETS,
                                       ordered=True)
    counts["Model"] = pd.Categorical(counts["Model"],
                                     categories=SEGMENT_COUNT_MODELS,
                                     ordered=True)
    return (counts.dropna(subset=["N_Segments"])
            .sort_values(["Dataset", "Model", "Cell"]).reset_index(drop=True))


SUMMARY_METRICS = [COSINE, KAPPA, JACCARD]

SUMMARY_METRIC_LABELS = {COSINE: COMPOSITION_DISPLAY, KAPPA: KAPPA_DISPLAY,
                         JACCARD: JACCARD_DISPLAY}
SUMMARY_METRIC_ORDER = [SUMMARY_METRIC_LABELS[m] for m in SUMMARY_METRICS]

SUMMARY_DOMAINS = [FULL, NOQH]

SUMMARY_COMPARISONS = ["cross_sample", "replicates", "joint_indiv"]

COMPARISON_ROW_COLUMNS = ["Dataset", "Cell", "Comparison", "Domain", "Metric",
                          "Method", "Value"]


def _summary_categoricals(rows):
    """The dataset, method, domain, metric and comparison columns as ordered
    categoricals."""
    rows = rows.copy()
    rows["Dataset"] = pd.Categorical(rows["Dataset"], categories=DATASETS,
                                     ordered=True)
    rows["Method"] = pd.Categorical(rows["Method"], categories=METHODS,
                                    ordered=True)
    if "Domain" in rows:
        rows["Domain"] = pd.Categorical(rows["Domain"],
                                        categories=SUMMARY_DOMAINS, ordered=True)
    if "Metric" in rows:
        rows["Metric"] = pd.Categorical(rows["Metric"],
                                        categories=SUMMARY_METRIC_ORDER,
                                        ordered=True)
    if "Comparison" in rows:
        rows["Comparison"] = pd.Categorical(rows["Comparison"],
                                            categories=SUMMARY_COMPARISONS,
                                            ordered=True)
    return rows.dropna(subset=["Dataset", "Method"])


def _metric_value_rows(rows, comparison, dataset, domain, where,
                       method_col="Method", unit_col=None, wide=False):
    """Every metric of `rows` in long form, one row per comparison and
    metric."""
    frames = []
    for metric in SUMMARY_METRICS:
        col = _metric_column(rows, metric, where, domain if wide else None)
        if col is None:
            continue
        keys = rows[method_col].map(canonical)
        keep = keys.notna() & rows[col].notna()
        if not keep.any():
            continue
        frames.append(pd.DataFrame({
            "Dataset": dataset,
            "Cell": (rows.loc[keep, unit_col].astype(str) if unit_col
                     else str(dataset)),
            "Comparison": comparison,
            "Domain": domain,
            "Metric": SUMMARY_METRIC_LABELS[metric],
            "Method": keys[keep].values,
            "Value": rows.loc[keep, col].astype(float).values}))
    return frames


def _long_metric_rows(df, comparison, dataset, where, **kwargs):
    """_metric_value_rows() off a cache with the domain in a Mode column."""
    frames = []
    for domain in SUMMARY_DOMAINS:
        rows = _domain_rows(df, domain, where)
        if rows is not None:
            frames += _metric_value_rows(rows, comparison, dataset, domain,
                                         where, **kwargs)
    return frames


def _encode_comparison_rows(workdir, excluded):
    """The three comparisons of ENCODE, gated as collect_encode() gates
    them."""
    out = os.path.join(os.path.expanduser(workdir), "out")
    frames = []

    rep = _drop_excluded(_read(f"{out}/df_joint_rep.pkl"), excluded)
    if rep is not None:
        frames += _long_metric_rows(rep, "replicates", "encode",
                                    "ENCODE df_joint_rep.pkl",
                                    unit_col="Dataset")

    pairs = _read(f"{out}/comparison_table.tsv")
    if pairs is not None:
        mint_a = pairs["ds_a"].astype(str).str.endswith("_mint")
        mint_b = pairs["ds_b"].astype(str).str.endswith("_mint")
        pairs = _drop_excluded(pairs[mint_a == mint_b], excluded,
                               method_col="method",
                               dataset_cols=("ds_a", "ds_b"))
        pairs = pairs.assign(Pair=pairs["ds_a"].astype(str) + "/"
                                  + pairs["ds_b"].astype(str))
        for domain in SUMMARY_DOMAINS:
            frames += _metric_value_rows(
                pairs, "cross_sample", "encode", domain,
                "ENCODE comparison_table.tsv", method_col="method",
                unit_col="Pair", wide=True)

    cross_sample = _drop_excluded(_read(f"{out}/df_cross_sample.pkl"), excluded)
    if cross_sample is not None:
        joint_rows = cross_sample[
            cross_sample["SameAssay"].fillna(True).astype(bool)
            & cross_sample["Method"].map(canonical).map(is_joint)]
        frames += _long_metric_rows(joint_rows, "cross_sample", "encode",
                                    "ENCODE df_cross_sample.pkl",
                                    unit_col="Dataset")

    joint = _drop_excluded(_read(f"{out}/df_joint_indiv.pkl"),
                           _excluded_comparisons(excluded))
    if joint is not None:
        frames += _long_metric_rows(joint, "joint_indiv", "encode",
                                    "ENCODE df_joint_indiv.pkl",
                                    unit_col="Dataset")
    return frames


SAGACONF_COMPARISON_CACHES = {
    "replicates": {FULL: "df_rep.pkl", NOQH: "df_rep_noqh.pkl"},
    "cross_sample": {FULL: "df_cross_sample.pkl",
                     NOQH: "df_cross_sample_noqh.pkl"},
    "joint_indiv": {FULL: "df_ji.pkl", NOQH: "df_ji_noqh.pkl"},
}


def _sagaconf_comparison_rows(workdir, excluded):
    out = os.path.join(os.path.expanduser(workdir), "out")
    frames = []
    for comparison, caches in SAGACONF_COMPARISON_CACHES.items():
        drop = (_excluded_comparisons(excluded) if comparison == "joint_indiv"
                else excluded)
        for domain, cache in caches.items():
            rows = _drop_excluded(_read(f"{out}/{cache}"), drop)
            if rows is None:
                continue
            frames += _metric_value_rows(rows, comparison, "sagaconf", domain,
                                         f"SAGAconf {cache}",
                                         unit_col="Dataset")
    return frames


def _epi1000_comparison_rows(workdir, excluded):
    """The three comparisons of the 1000 epigenomes; no gate, as in
    collect_epi1000()."""
    out = os.path.join(os.path.expanduser(workdir), "out")
    frames = []

    rep = _drop_reference_indiv(_read(f"{out}/df_replicates.csv"),
                                "1000 epigenomes df_replicates.csv")
    if rep is not None:
        rep = rep.assign(Pair=rep["EID1"].astype(str) + "/"
                              + rep["EID2"].astype(str))
        frames += _long_metric_rows(rep, "replicates", "epi1000",
                                    "1000 epigenomes df_replicates.csv",
                                    unit_col="Pair")

    pairs = _cross_sample_epi1000(out)
    if pairs is not None:
        frames += _long_metric_rows(pairs, "cross_sample", "epi1000",
                                    "1000 epigenomes pw_cache_*.pkl")

    joint = _read(f"{out}/df_joint_indiv_comparison.csv")
    if joint is not None:
        frames += _long_metric_rows(
            joint, "joint_indiv", "epi1000",
            "1000 epigenomes df_joint_indiv_comparison.csv",
            unit_col="Dataset")
    return frames


COMPARISON_COLLECTORS = {"encode": _encode_comparison_rows,
                         "sagaconf": _sagaconf_comparison_rows,
                         "epi1000": _epi1000_comparison_rows}


def collect_comparisons(workdirs, incomplete):
    """Every comparison, metric and domain, one row per measured comparison."""
    excluded = _excluded_pairs(incomplete, announce=False)
    frames = []
    for dataset, collect in COMPARISON_COLLECTORS.items():
        if dataset in workdirs:
            frames += collect(workdirs[dataset], excluded)
    if not frames:
        print("  no comparison metrics collected")
        return pd.DataFrame(columns=COMPARISON_ROW_COLUMNS)
    rows = pd.concat(frames, ignore_index=True)[COMPARISON_ROW_COLUMNS]
    return _summary_categoricals(rows).sort_values(
        ["Comparison", "Domain", "Metric", "Dataset", "Method"]
    ).reset_index(drop=True)


ENTROPY_ROW_COLUMNS = ["Dataset", "Cell", "Domain", "Method", "Value"]


def _entropy_rows(rows, dataset, domain, method_col, value_col, unit_col):
    keys = rows[method_col].map(canonical)
    keep = keys.notna() & rows[value_col].notna()
    if not keep.any():
        return []
    return [pd.DataFrame({"Dataset": dataset,
                          "Cell": rows.loc[keep, unit_col].astype(str),
                          "Domain": domain,
                          "Method": keys[keep].values,
                          "Value": rows.loc[keep, value_col].astype(float).values})]


def collect_entropy(workdirs, incomplete):
    """The transition matrix entropy of every segmentation, over both
    domains."""
    excluded = _excluded_pairs(incomplete, announce=False)
    frames = []

    if "encode" in workdirs:
        out = os.path.join(os.path.expanduser(workdirs["encode"]), "out")
        for domain in SUMMARY_DOMAINS:
            suffix = "" if domain == FULL else f"_{domain}"
            entropy = _drop_excluded(
                _encode_frames(out, f"entropy_summary{suffix}.tsv",
                               ["total_entropy"]), excluded)
            if entropy is None:
                _no_cache("ENCODE", f"entropy_summary{suffix}.tsv")
                continue
            frames += _entropy_rows(entropy, "encode", domain, "Method",
                                    "total_entropy", "Segmentation")

    if "sagaconf" in workdirs:
        out = os.path.join(os.path.expanduser(workdirs["sagaconf"]), "out")
        for domain, cache in {FULL: "df_entropy.pkl",
                              NOQH: "df_entropy_noqh.pkl"}.items():
            entropy = _drop_excluded(_read(f"{out}/{cache}"), excluded)
            if entropy is None:
                continue
            frames += _entropy_rows(entropy, "sagaconf", domain, "Method",
                                    "total_entropy", "Dataset")

    if "epi1000" in workdirs:
        out = os.path.join(os.path.expanduser(workdirs["epi1000"]), "out")
        results = _read(f"{out}/df_results.csv")
        for domain in SUMMARY_DOMAINS:
            if results is None:
                break
            col = _metric_column(results, "Entropy",
                                 "1000 epigenomes df_results.csv", domain)
            if col is None:
                continue
            frames += _entropy_rows(results, "epi1000", domain, "Method", col,
                                    "Dataset")
        entropy_15 = _reference_joint_only(_read(f"{out}/df_entropy_15.csv"),
                                           "1000 epigenomes df_entropy_15.csv")
        if entropy_15 is not None:
            for domain in SUMMARY_DOMAINS:
                rows = _domain_rows(entropy_15, domain,
                                    "1000 epigenomes df_entropy_15.csv")
                if rows is not None:
                    frames += _entropy_rows(rows, "epi1000", domain, "Method",
                                            "Entropy", "Dataset")

    if not frames:
        print("  no entropy collected")
        return pd.DataFrame(columns=ENTROPY_ROW_COLUMNS)
    rows = pd.concat(frames, ignore_index=True)[ENTROPY_ROW_COLUMNS]
    return _summary_categoricals(rows).sort_values(
        ["Domain", "Dataset", "Method"]).reset_index(drop=True)


PEAK_MODELS = [CHROMHMM_DEFAULT, KMEANS_HOMER, BMM3_HOMER, KMEANS_MACS2, BMM3_MACS2,
               KMEANS_OMNI, BMM3_OMNI]

PEAK_MARKS = ["H3K4me3", "H3K4me1", "H3K27ac", "H3K27me3", "H3K36me3",
              "H3K9me3"]

PEAK_ROW_COLUMNS = ["Dataset", "Cell", "Mark", "Method", "N_Peaks",
                    "Mean_Length"]


def _peak_method(name):
    """Canonical model key of a peak-table method; 'Default' is the ChromHMM
    binarization."""
    if str(name).strip().lower() == "default":
        return CHROMHMM_DEFAULT
    return canonical(name)


def _peak_frame(df, dataset, dataset_col):
    return pd.DataFrame({"Dataset": dataset,
                         "Cell": df[dataset_col].astype(str),
                         "Mark": df["mark"].astype(str),
                         "Method": df["method"].map(_peak_method),
                         "N_Peaks": df["n_peaks"].astype(float),
                         "Mean_Length": df["mean_length"].astype(float)})


def _encode_peak_stats(workdir):
    """The peak table of every ENCODE dataset, which sits next to the
    segmentations."""
    root = os.path.expanduser(workdir)
    frames = []
    for ds in ENCODE_DATASETS:
        path = os.path.join(root, ds, "peaks", "peak_stats.tsv")
        if not os.path.exists(path):
            continue
        frames.append(_peak_frame(_read(path).assign(Dataset=ds), "encode",
                                  "Dataset"))
    if not frames:
        _no_cache("ENCODE", "peaks/peak_stats.tsv")
        return []
    return frames


def _drop_unanalyzed_samples(rows):
    """`rows` without the samples whose peak table is zero everywhere - a run
    that never happened, unlike a single empty call, which stays."""
    totals = rows.groupby(["Dataset", "Cell"])["N_Peaks"].sum()
    empty = set(totals[totals <= 0].index)
    if empty:
        print("  no peaks at all for "
              + ", ".join(f"{DATASET_NAMES[ds]}/{cell}"
                          for ds, cell in sorted(empty))
              + " - the peak analysis of those samples did not run, so they "
                "are left out of the peak figures")
        rows = rows[[(ds, cell) not in empty
                     for ds, cell in zip(rows["Dataset"], rows["Cell"])]]
    return rows


def collect_peak_stats(workdirs):
    """Peak count and mean peak length of every (sample, mark), over
    PEAK_MARKS."""
    frames = []
    if "encode" in workdirs:
        frames += _encode_peak_stats(workdirs["encode"])
    if "sagaconf" in workdirs:
        out = os.path.join(os.path.expanduser(workdirs["sagaconf"]), "out")
        peaks = _read(f"{out}/df_peaks.pkl")
        if peaks is not None:
            frames.append(_peak_frame(peaks, "sagaconf", "Dataset"))
    if "epi1000" in workdirs:
        out = os.path.join(os.path.expanduser(workdirs["epi1000"]), "out")
        peaks = _read(f"{out}/df_peaks.csv")
        if peaks is not None:
            frames.append(_peak_frame(peaks, "epi1000", "dataset"))
    if not frames:
        print("  no peak statistics collected")
        return pd.DataFrame(columns=PEAK_ROW_COLUMNS)
    rows = pd.concat(frames, ignore_index=True)
    rows = rows[rows["Mark"].isin(PEAK_MARKS) & rows["Method"].isin(PEAK_MODELS)]
    rows = _drop_unanalyzed_samples(rows)
    rows["Mark"] = pd.Categorical(rows["Mark"], categories=PEAK_MARKS,
                                  ordered=True)
    return _summary_categoricals(rows[PEAK_ROW_COLUMNS]).sort_values(
        ["Dataset", "Method", "Cell", "Mark"]).reset_index(drop=True)


# The quiescent state is the "nothing here" label - no mark reaches it, and in
# most samples it is the majority of the genome. What is left of the genome
# once it is taken out is what a segmentation actually annotates, so its share
# is read here for every segmentation of every dataset, next to the segment
# counts above.
#
# The share is of the genome the segmentation covers, which is what the cached
# compositions are normalized by; ENCODE also carries the covered length, and
# there every complete segmentation covers the same 3.09 Gb, so the two
# denominators are one. A state matching left unlabelled ("Unknown", under 2%
# of the ENCODE Mint-ChIP segmentations and absent everywhere else) is counted
# as annotated - it is not the quiescent state, only an unmatched one.
ANNOTATED_COLUMNS = ["Dataset", "Cell", "Scope", "Segmentation", "Model",
                     "Covered_bp", "Quiescent", "Annotated"]


def _annotated_row(dataset, scope, segmentation, model, states, weights,
                   covered=np.nan):
    """One row off the per-state weights of a segmentation, bp or fractions."""
    weights = np.asarray(weights, dtype=float)
    total = weights.sum()
    if not np.isfinite(total) or total <= 0:
        return None
    quiescent = weights[[utils.is_quiescent(s) for s in states]].sum() / total
    return {"Dataset": dataset, "Scope": scope, "Segmentation": segmentation,
            "Model": model, "Covered_bp": covered,
            "Quiescent": quiescent, "Annotated": 1.0 - quiescent}


def _encode_annotated_share(workdir, excluded):
    """Quiescent and annotated share of every ENCODE segmentation, reference
    included, off the per-state bp of its report.tsv; both scopes, kept apart
    in Scope."""
    out = os.path.join(os.path.expanduser(workdir), "out")
    references = _encode_reference_ids()
    rows = []
    for pattern, scope in ((f"{out}/*/ref/report.tsv", "pooled"),
                           (f"{out}/*/matched/*/report.tsv", "pooled"),
                           (f"{out}/*/rep*/matched/*/report.tsv", "replicates")):
        for path in sorted(glob.glob(pattern)):
            dirpath = os.path.dirname(path)
            parts = os.path.relpath(dirpath, out).split(os.sep)
            ds, leaf = parts[0], parts[-1]
            if ds not in ENCODE_DATASETS:
                continue
            if leaf == "ref":
                model, name = REFERENCE, f"{references.get(ds)}_chromhmm"
            else:
                model = canonical(leaf)
                name = leaf + (f"_{parts[1]}" if scope == "replicates" else "")
            if model is None:
                continue
            report = _read(path)
            if report is None:
                continue
            row = _annotated_row(ds, scope, f"{ds}:{name}", model,
                                 report["state"], report["total_bp"],
                                 covered=float(report["total_bp"].sum()))
            if row is not None:
                rows.append(row)
    if not rows:
        _no_cache("ENCODE", "report.tsv per-state coverage")
        return None
    return _drop_excluded(pd.DataFrame(rows), excluded, method_col="Model")


def _sagaconf_annotated_share(workdir, excluded):
    """The same for SAGAconf, off the interpreted state types: its states are
    matched per dataset and numbered, so which of them is the quiescent one is
    a question about the emissions, not about the name."""
    out = os.path.join(os.path.expanduser(workdir), "out")
    interp = _read(f"{out}/df_interp_matched.pkl")
    if interp is None:
        _no_cache("SAGAconf", "df_interp_matched.pkl (interpreted state types)")
        return None
    rows = []
    for (ds, method, rep), group in interp.groupby(["Dataset", "Method", "Rep"]):
        model = canonical(method)
        if model is None:
            continue
        row = _annotated_row(ds, "", f"{ds}:{model}_{rep}", model,
                             group["Type"], group["Fraction"])
        if row is not None:
            rows.append(row)
    if not rows:
        return None
    return _drop_excluded(pd.DataFrame(rows), excluded, method_col="Model")


def _epi1000_annotated_share(workdir, excluded):
    """The same for the 1000 epigenomes, one row per epigenome and model. The
    published 15-state joint model is the joint ChromHMM of this dataset and
    is cached apart from the de-novo compositions."""
    out = os.path.join(os.path.expanduser(workdir), "out")
    rows = []
    comp = _read(f"{out}/df_comp.csv")
    if comp is not None:
        for (ds, method), group in comp.groupby(["Dataset", "Method"]):
            model = canonical(method)
            if model is None:
                continue
            row = _annotated_row(ds, "", f"{ds}:{model}", model,
                                 group["State"], group["Fraction"])
            if row is not None:
                rows.append(row)
    published = _read(f"{out}/stats_15_cache.pkl")
    if published is not None:
        for (ds, method), stats in published.items():
            if canonical(method) != JOINT_CHROMHMM:
                continue
            states = pd.DataFrame(stats["composition"])
            row = _annotated_row(ds, "", f"{ds}:{JOINT_CHROMHMM}",
                                 JOINT_CHROMHMM, states["State"],
                                 states["Fraction"])
            if row is not None:
                rows.append(row)
    if not rows:
        _no_cache("1000 epigenomes", "df_comp.csv (state composition)")
        return None
    return _drop_excluded(pd.DataFrame(rows), excluded, method_col="Model")


ANNOTATED_COLLECTORS = {"encode": _encode_annotated_share,
                        "sagaconf": _sagaconf_annotated_share,
                        "epi1000": _epi1000_annotated_share}


def collect_annotated_share(workdirs, incomplete):
    """One row per segmentation of every dataset: how much of the genome it
    puts on a state other than the quiescent one."""
    excluded = _excluded_pairs(incomplete, announce=False)
    frames = []
    for dataset, collect in ANNOTATED_COLLECTORS.items():
        if dataset not in workdirs:
            continue
        shares = collect(workdirs[dataset], excluded)
        if shares is None or shares.empty:
            continue
        shares = shares[shares["Model"].isin(SEGMENT_COUNT_MODELS)]
        frames.append(shares.assign(Cell=shares["Dataset"].astype(str),
                                    Dataset=dataset)
                      .reindex(columns=ANNOTATED_COLUMNS))
    if not frames:
        print("  no state coverage collected")
        return pd.DataFrame(columns=ANNOTATED_COLUMNS)
    shares = pd.concat(frames, ignore_index=True)
    shares["Dataset"] = pd.Categorical(shares["Dataset"], categories=DATASETS,
                                       ordered=True)
    shares["Model"] = pd.Categorical(shares["Model"],
                                     categories=SEGMENT_COUNT_MODELS,
                                     ordered=True)
    return (shares.dropna(subset=["Annotated"])
            .sort_values(["Dataset", "Model", "Cell"]).reset_index(drop=True))


## Scoring

In [ ]:
def score_evidence(df):
    """A rank score in [0, 1] per evidence row, within its family and (axis,
    cell) group."""
    out = df.copy()
    out["score"] = np.nan
    families = out["Method"].astype(str).map(is_joint)
    for (axis, cell, _), group in out.groupby(["Axis", "Cell", families],
                                              observed=True):
        values = group["Value"].astype(float)
        n = len(values)
        if n == 1:
            out.loc[group.index, "score"] = 0.5
            continue
        ranks = values.rank(ascending=AXES[str(axis)][1] == "low", method="average")
        out.loc[group.index, "score"] = (n - ranks) / (n - 1)
    return out


def _axis_table(scored, aggfunc, fold, blank, column="score"):
    """The Method x AXIS_ORDER table of `aggfunc` over `column`."""
    table = scored.pivot_table(index="Method", columns="Axis", values=column,
                               aggfunc=aggfunc, observed=True)
    for main in VARIANT_MAIN_AXES:
        avail = [v for v in axis_variants(main) if v in table.columns]
        if avail:
            table[main] = fold(table[avail])
    table = table.reindex(index=METHODS, columns=AXIS_ORDER)
    joint = np.array([is_joint(m) for m in table.index])
    excluded = [a for a in table.columns
                if str(a).startswith(("joint_indiv", "cross_assay"))]
    table.loc[joint, excluded] = blank
    return table


def axis_scores(scored):
    return _axis_table(scored, "mean", lambda t: t.mean(axis=1), np.nan)


def axis_values(scored):
    """The Method x AXIS_ORDER table of the raw evidence values."""
    return _axis_table(scored, "mean", lambda t: t.mean(axis=1), np.nan,
                       column="Raw")


def count_cells(scored):
    """The dataset cells behind each score, the largest of a main axis'
    variants."""
    return _axis_table(scored, "size", lambda t: t.max(axis=1),
                       0).fillna(0).astype(int)


COVERED, MISSING = "yes", "no"


NO_DOMAIN = "-"


def axis_domains(axis):
    """Display spelling of the comparison domains `axis` is scored over."""
    if axis not in VARIANT_MAIN_AXES:
        return NO_DOMAIN
    domains = []
    for variant in axis_variants(axis):
        domain = utils.normalize_domain(variant.rsplit("_", 1)[-1])
        if domain is not None and domain not in domains:
            domains.append(domain)
    return " / ".join(utils.domain_display(d) for d in domains) or NO_DOMAIN


def feature_coverage(evidence):
    """Which features each dataset contributes evidence for, per main axis."""
    present = set(zip(evidence["Axis"].astype(str), evidence["Dataset"]))
    rows = []
    for axis in AXIS_ORDER:
        variants = axis_variants(axis) if axis in VARIANT_MAIN_AXES else [axis]
        row = {"Feature": axis_label(axis, oneline=True),
               "Metric": AXES[axis][2],
               "Domain": axis_domains(axis)}
        for dataset in DATASETS:
            have = sum((v, dataset) in present for v in variants)
            row[DATASET_NAMES[dataset]] = (
                COVERED if have == len(variants)
                else MISSING if have == 0
                else f"{have}/{len(variants)}")
        rows.append(row)
    return pd.DataFrame(rows).set_index("Feature")


def dataset_scores(scored):
    return (scored.pivot_table(index="Method", columns="Dataset", values="score",
                               aggfunc="mean", observed=True)
            .reindex(index=METHODS, columns=DATASETS))


def summary_ranking(scored, methods, table=None):
    """Rank `methods` among themselves over the axes that group is scored
    on."""
    table = axis_scores(scored) if table is None else table
    table = table.loc[[m for m in methods if m in table.index]]
    table = table[[a for a in table.columns if table[a].notna().any()]]
    weights = np.array([AXIS_WEIGHTS.get(a, 1.0) for a in table.columns])
    scores = (table * weights).sum(axis=1) / (table.notna() * weights).sum(axis=1)
    ranking = pd.DataFrame({
        "Method": [utils.display_name(m) for m in table.index],
        "Score": scores,
        "Axes": table.notna().sum(axis=1),
        "Datasets": [scored.loc[scored["Method"] == m, "Dataset"].nunique()
                     for m in table.index],
        "Best axis": [axis_label(table.columns[int(np.nanargmax(row))], oneline=True)
                      if row.notna().any() else "-"
                      for _, row in table.iterrows()],
        "Worst axis": [axis_label(table.columns[int(np.nanargmin(row))], oneline=True)
                       if row.notna().any() else "-"
                       for _, row in table.iterrows()],
    }, index=table.index)
    ranking = ranking.sort_values("Score", ascending=False)
    ranking.insert(0, "Rank", range(1, len(ranking) + 1))
    return ranking


## Evidence and scores

In [ ]:
df_evidence, df_incomplete = build_evidence(WORKDIRS)
df_scored = score_evidence(df_evidence)

table_scores = axis_scores(df_scored)
table_values = axis_values(df_scored)
table_counts = count_cells(df_scored)
table_datasets = dataset_scores(df_scored)
table_coverage = feature_coverage(df_evidence)
df_ranking_individual = summary_ranking(df_scored, INDIVIDUAL_METHODS,
                                        table_scores)
df_ranking_joint = summary_ranking(df_scored, JOINT_METHODS, table_scores)
df_ranking_all = summary_ranking(df_scored, METHODS, table_scores)

df_segment_counts = collect_segment_counts(WORKDIRS, df_incomplete)
df_annotated = collect_annotated_share(WORKDIRS, df_incomplete)
df_comparisons = collect_comparisons(WORKDIRS, df_incomplete)
df_entropy_summary = collect_entropy(WORKDIRS, df_incomplete)
df_peak_stats = collect_peak_stats(WORKDIRS)

for name, frame, index in [("evidence", df_scored, False),
                           ("axis_scores", table_scores, True),
                           ("axis_values", table_values, True),
                           ("axis_cells", table_counts, True),
                           ("dataset_scores", table_datasets, True),
                           ("feature_coverage", table_coverage, True),
                           ("ranking_individual", df_ranking_individual, True),
                           ("ranking_joint", df_ranking_joint, True),
                           ("ranking_all", df_ranking_all, True),
                           ("incomplete_segmentations", df_incomplete, False),
                           ("segment_counts", df_segment_counts, False),
                           ("annotated_share", df_annotated, False),
                           ("comparison_metrics", df_comparisons, False),
                           ("entropy_per_segmentation", df_entropy_summary, False),
                           ("peak_stats", df_peak_stats, False)]:
    frame.to_csv(f"{OUT}/{name}.tsv", sep="\t", index=index)

coverage = (df_evidence.assign(Axis=df_evidence["Axis"].astype(str))
            .pivot_table(index="Axis", columns="Dataset", values="Value",
                         aggfunc="size")
            .reindex(index=[a for a in ALL_AXIS_CATEGORIES
                            if a in set(df_evidence["Axis"].astype(str))],
                     columns=DATASETS)
            .fillna(0).astype(int))
coverage.to_csv(f"{OUT}/coverage.tsv", sep="\t")

print(f"\n{len(df_evidence)} evidence rows over "
      f"{df_evidence['Cell'].nunique()} cells, tables in {OUT}")
print(f"\nevidence rows per axis and dataset "
      f"({COMPARISON_METRICS} over {COMPARISON_DOMAINS}, "
      f"[{COSINE!r}] over {COSINE_DOMAINS}):")
print(coverage.to_string())

print(f"\nsegment counts of {len(df_segment_counts)} segmentations, mean per "
      f"model and dataset (\u00d710\u00b3):")
print((df_segment_counts.pivot_table(index="Model", columns="Dataset",
                                     values="N_Segments", aggfunc="mean",
                                     observed=True) / 1000).round(1).to_string())

print(f"\nannotated (non-quiescent) share of {len(df_annotated)} "
      f"segmentations, mean per model and dataset (%):")
print((df_annotated.pivot_table(index="Model", columns="Dataset",
                                values="Annotated", aggfunc="mean",
                                observed=True) * 100).round(1).to_string())

print(f"\n{len(df_comparisons)} comparison rows over "
      f"{df_comparisons['Comparison'].nunique()} comparisons x "
      f"{len(SUMMARY_METRIC_ORDER)} metrics x {len(SUMMARY_DOMAINS)} domains, "
      f"{len(df_entropy_summary)} entropy rows, {len(df_peak_stats)} peak rows")


# 2. Doing plotting

In [ ]:
SHORT_NAMES = {
    CHROMHMM_DEFAULT:   "C",
    KMEANS_HOMER:       "H",
    KMEANS_MACS2:       "M",
    KMEANS_OMNI:        "O",
    BMM3_HOMER:          "BH",
    BMM3_MACS2:          "BM",
    BMM3_OMNI:           "BO",
    JOINT_CHROMHMM:     "J-C",
    JOINT_KMEANS_HOMER: "J-H",
    JOINT_KMEANS_MACS2: "J-M",
    JOINT_KMEANS_OMNI:  "J-O",
    JOINT_BMM3_HOMER:    "J-BH",
    JOINT_BMM3_MACS2:    "J-BM",
    JOINT_BMM3_OMNI:     "J-BO",
}

LINE_STYLES = [":", "--", "-.", "-"]
MARKERS = ["o", "s", "^", "D"]

FIGURE_DPI = 200

POLAR_SIZE = (10.4, 8.4)
POLAR_TICKS = [0.25, 0.5, 0.75, 1.0]
POLAR_LIMIT = 1.12
POLAR_HOLE = -0.18
POLAR_LEGEND = dict(loc="upper left", bbox_to_anchor=(1.04, 1.0), fontsize=9,
                    frameon=False)
SEPARATOR_COLOR = "#dddddd"


def _polar_axes(fig, axes_keys, angles, sector, labels=None,
                ytick_labels=None):
    ax = fig.add_subplot(projection="polar")
    ax.set_theta_direction(-1)
    ax.set_theta_offset(np.pi / 2)
    ax.set_rorigin(POLAR_HOLE)
    ax.set_ylim(0, POLAR_LIMIT)
    ax.set_yticks(POLAR_TICKS)
    ax.set_yticklabels(ytick_labels or [f"{t:.2f}" for t in POLAR_TICKS],
                       fontsize=7, color="#777777")
    for label in ax.get_yticklabels():
        label.set_bbox(dict(facecolor="white", edgecolor="none", alpha=0.7,
                            boxstyle="round,pad=0.12"))
    ax.set_rlabel_position(np.degrees(sector / 2))
    ax.set_xticks(angles)
    ax.set_xticklabels(labels or [axis_label(a) for a in axes_keys], fontsize=8)
    ax.tick_params(axis="x", pad=14)
    ax.grid(color="#bbbbbb", linewidth=0.6, alpha=0.6)
    for spine in ax.spines.values():
        spine.set_visible(False)
    circle = np.linspace(0, 2 * np.pi, 361)
    for color, width, zorder in [("white", 3, 7), (SEPARATOR_COLOR, 0.8, 8)]:
        ax.plot(circle, np.zeros_like(circle), color=color, linewidth=width,
                zorder=zorder, clip_on=False)
    return ax


def plot_radar(table, outdir):
    for group, methods in [("individual", INDIVIDUAL_METHODS), ("joint", JOINT_METHODS)]:
        methods = [m for m in methods if m in table.index]
        axes_keys = [a for a in table.columns
                     if methods and table.loc[methods, a].notna().any()]

        if not axes_keys:
            continue

        sector = 2 * np.pi / len(axes_keys)
        angles = np.linspace(0, 2 * np.pi, len(axes_keys), endpoint=False)
        closed = np.concatenate([angles, angles[:1]])

        fig = plt.figure(figsize=POLAR_SIZE)
        ax = _polar_axes(fig, axes_keys, angles, sector)
        for i, method in enumerate(methods):
            values = table.loc[method, axes_keys].astype(float).values
            values_closed = np.concatenate([values, values[:1]])
            ax.plot(closed, values_closed,
                    color=utils.method_color(method),
                    linestyle=LINE_STYLES[i % len(LINE_STYLES)],
                    marker=MARKERS[i % len(MARKERS)], markersize=5,
                    linewidth=2.0, zorder=2,
                    label=f"{SHORT_NAMES[method]}  {utils.display_name(method)}")
            if not np.isnan(values_closed).any():
                fill = ax.fill(closed, values_closed,
                               color=utils.method_color(method), alpha=0.07,
                               zorder=1)
                if is_joint(method):
                    for poly in fill:
                        poly.set_hatch(JOINT_HATCH)
        ax.set_title(f"Rank profile of the {group} models",
                     fontsize=12, fontweight="bold", pad=32)
        ax.legend(**POLAR_LEGEND)
        fig.text(0.5, 0.03, f"radius = mean rank score over the dataset cells "
                            f"of that axis, 1 = best of the {group} models",
                 ha="center", fontsize=8, color="#555555")
        utils.save_fig(fig, os.path.join(outdir, f"radar_{group}.png"),
                       tight=False, dpi=FIGURE_DPI)


def _fmt_ends(inner, outer):
    """The two ends of a spoke, formatted alike so they read as one range."""
    finite = [v for v in (inner, outer) if np.isfinite(v)]
    if not finite:
        return "-", "-"
    scale = max(abs(v) for v in finite)
    digits = 3 if scale < 10 else 2 if scale < 1000 else 0
    return tuple(f"{v:.{digits}f}" if np.isfinite(v) else "-"
                 for v in (inner, outer))


def _spoke_ends(values, better):
    """(worst, best) raw value of one spoke, in the direction it improves
    in."""
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if not len(finite):
        return np.nan, np.nan
    lo, hi = float(finite.min()), float(finite.max())
    return (hi, lo) if better == "low" else (lo, hi)


def _spoke_scale(axis):
    """(inner, outer) of one spoke of the value radar, outward = better."""
    return (1.0, 0.0) if AXES[str(axis)][1] == "low" else (0.0, 1.0)


def _spoke_radius(values, inner, outer):
    """`values` as a position in [0, 1] between the two ends of their spoke."""
    return (np.asarray(values, dtype=float) - inner) / (outer - inner)


def plot_radar_values(table, outdir):
    """Radar of the raw axis values, every spoke on the [0, 1] of its
    metric."""
    for group, methods in [("individual", INDIVIDUAL_METHODS), ("joint", JOINT_METHODS)]:
        methods = [m for m in methods if m in table.index]
        axes_keys = [a for a in table.columns
                     if methods and table.loc[methods, a].notna().any()]
        if not axes_keys:
            continue

        raw = table.loc[methods, axes_keys].astype(float)
        scale = {a: _spoke_scale(a) for a in axes_keys}
        ends = {a: _spoke_ends(raw[a].values, AXES[str(a)][1])
                for a in axes_keys}
        radius = pd.DataFrame({a: _spoke_radius(raw[a].values,
                                                scale[a][0],
                                                scale[a][1])
                               for a in axes_keys}, index=methods)
        for a in axes_keys:
            if str(a) not in UNBOUNDED_AXES:
                continue
            over = [m for m in methods if raw.loc[m, a] > 1]
            if over:
                print(f"  {axis_label(a, oneline=True)} is above the [0, 1] "
                      f"radius for {', '.join(utils.display_name(m) for m in over)}"
                      f" - drawn on the hole, read the printed range instead")

        sector = 2 * np.pi / len(axes_keys)
        angles = np.linspace(0, 2 * np.pi, len(axes_keys), endpoint=False)
        closed = np.concatenate([angles, angles[:1]])
        labels = []
        for a in axes_keys:
            worst, best = _fmt_ends(*ends[a])
            labels.append(f"{axis_label(a)}\n{worst} \u2192 {best}")

        fig = plt.figure(figsize=POLAR_SIZE)
        ax = _polar_axes(fig, axes_keys, angles, sector, labels=labels)
        for i, method in enumerate(methods):
            values = radius.loc[method, axes_keys].astype(float).values
            values_closed = np.concatenate([values, values[:1]])
            ax.plot(closed, values_closed,
                    color=utils.method_color(method),
                    linestyle=LINE_STYLES[i % len(LINE_STYLES)],
                    marker=MARKERS[i % len(MARKERS)], markersize=5,
                    linewidth=2.0, zorder=2,
                    label=f"{SHORT_NAMES[method]}  {utils.display_name(method)}")
            if not np.isnan(values_closed).any():
                fill = ax.fill(closed, values_closed,
                               color=utils.method_color(method), alpha=0.07,
                               zorder=1)
                if is_joint(method):
                    for poly in fill:
                        poly.set_hatch(JOINT_HATCH)
        ax.set_title(f"Value profile of the {group} models",
                     fontsize=12, fontweight="bold", pad=32)
        ax.legend(**POLAR_LEGEND)
        fig.subplots_adjust(bottom=0.13)
        fig.text(0.5, 0.012,
                 "radius = the value itself, on a fixed [0, 1] for every "
                 "axis; outward is always better\n"
                 f"under each axis: the worst \u2192 best value of the {group} "
                 f"models, the span the polygons cover on that scale",
                 ha="center", va="bottom", fontsize=8, color="#555555",
                 linespacing=1.6)
        utils.save_fig(fig, os.path.join(outdir, f"radar_values_{group}.png"),
                       tight=False, dpi=FIGURE_DPI)


LOWER_BETTER_MARK = " \u2193"


def _barplot_axis_label(axis, mark_low=False):
    """The one-line axis label a barplot puts on the x axis."""
    label = axis_label(axis, oneline=True)
    return label + LOWER_BETTER_MARK if mark_low and AXES[str(axis)][1] == "low" \
        else label


def _barplot_frame(scored, methods, axes_keys, column, mark_low=False):
    """The evidence of `axes_keys` for `methods` as one long frame."""
    rows = []
    for main in axes_keys:
        if main in VARIANT_MAIN_AXES:
            variants = [v for v in axis_variants(main)
                        if v in scored["Axis"].unique()]
            if not variants:
                continue
            df_main = scored[scored["Axis"].isin(variants) &
                             scored["Method"].isin(methods)].copy()
            df_main["Axis"] = main
        else:
            df_main = scored[(scored["Axis"] == main) &
                             scored["Method"].isin(methods)]
        rows.append(df_main[["Method", "Dataset", column, "Axis"]])

    if not rows:
        return None
    df_plot = pd.concat(rows, ignore_index=True)
    df_plot["Method"] = df_plot["Method"].astype(str)
    df_plot["Axis"] = df_plot["Axis"].astype(str)
    df_plot["Method Name"] = df_plot["Method"].map(utils.display_name)
    df_plot["Axis Label"] = df_plot["Axis"].map(
        lambda a: _barplot_axis_label(a, mark_low))
    return df_plot


def _axis_barplot(df_plot, methods, axes_keys, column, outpath,
                  title, xlabel, ylabel, mark_low=False):
    """One grouped barplot of `column` over the axes, a bar per method."""
    present = set(df_plot["Method"])
    order = [utils.display_name(m) for m in methods if str(m) in present]
    axes_labels = [_barplot_axis_label(a, mark_low) for a in axes_keys]

    fig, ax = plt.subplots(figsize=(max(8, len(axes_labels) * 1.5), 5))
    sns.barplot(data=df_plot, x="Axis Label", y=column, hue="Method Name",
                order=axes_labels, hue_order=order,
                palette={utils.display_name(m): utils.method_color(m) for m in methods},
                capsize=0.05, errorbar="se", err_kws={"linewidth": 1.0},
                edgecolor="lightgrey", linewidth=0.5, ax=ax)
    utils.hatch_joint(ax, order)
    utils.strip_points(ax, data=df_plot, x="Axis Label", y=column, hue="Method Name",
                       order=axes_labels, hue_order=order, size=3)

    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    ax.legend(title="Method", fontsize=8, title_fontsize=9,
              bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
    utils.save_fig(fig, outpath)


def _barplot_groups(table):
    """(group, methods, axes_keys) per model family."""
    for group, methods in [("all", METHODS)]:
        methods = [m for m in methods if m in table.index]
        axes_keys = [a for a in table.columns
                     if methods and table.loc[methods, a].notna().any()]
        if axes_keys:
            yield group, methods, axes_keys


def plot_radar_barplots(scored, table, outdir):
    """The rank radar as a barplot: the spread behind every spoke."""
    for group, methods, axes_keys in _barplot_groups(table):
        df_plot = _barplot_frame(scored, methods, axes_keys, "score")
        if df_plot is None:
            continue
        _axis_barplot(df_plot, methods, axes_keys, "score",
                      os.path.join(outdir, f"radar_barplot_{group}.png"),
                      f"Rank distribution of the {group} models",
                      "Ranking axis", "Rank score (1 = best)")


def plot_radar_value_barplots(scored, table, outdir):
    """The value radar as a barplot: the spread behind every spoke."""
    for group, methods, axes_keys in _barplot_groups(table):
        df_plot = _barplot_frame(scored, methods, axes_keys, "Raw",
                                 mark_low=True)
        if df_plot is None:
            continue
        _axis_barplot(df_plot, methods, axes_keys, "Raw",
                      os.path.join(outdir, f"radar_values_barplot_{group}.png"),
                      f"Value distribution of the {group} models",
                      "Ranking axis", "Axis value (\u2193 = lower is better)",
                      mark_low=True)


def plot_ranking(ranking, table, group, outpath):
    order = ranking.index.tolist()
    axes_keys = [a for a in table.columns if table.loc[order, a].notna().any()]
    fig, ax = plt.subplots(figsize=(8.4, 0.32 * len(order) + 0.8))
    positions = np.arange(len(order))
    bars = ax.barh(positions, ranking.loc[order, "Score"].values,
                   color=[utils.method_color(m) for m in order],
                   edgecolor="lightgrey", linewidth=1, height=0.5, zorder=2)
    for bar, method in zip(bars, order):
        if is_joint(method):
            bar.set_hatch(JOINT_HATCH)
    for y, method in enumerate(order):
        scores = table.loc[method, axes_keys].dropna().values
        rng = np.random.default_rng(y)
        ax.scatter(scores, y + rng.uniform(-0.1, 0.1, len(scores)),
                   s=POINT_SIZE, color="#333333", alpha=0.75,
                   linewidth=0.3, edgecolor="white", zorder=5)
        ax.text(1.04, y, f"{ranking.loc[method, 'Score']:.2f}",
                va="center", fontsize=7, color="#333333")
    ax.set_yticks(positions)
    ax.set_yticklabels([f"{ranking.loc[m, 'Rank']}. {utils.display_name(m)}"
                        for m in order], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.2)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.axvline(1.0, color="#cccccc", linewidth=0.8)
    ax.axvline(0.5, color="#999999", linewidth=0.8, linestyle="--")
    ax.set_xlabel(f"mean rank score over the axes (1 = best of the {group} "
                  f"models), points are the per-axis scores behind it",
                  fontsize=9)
    n_datasets = int(ranking["Datasets"].max()) if not ranking.empty else 0
    ax.set_title(f"Summary ranking of the {group} models "
                 f"({len(axes_keys)} axes, {n_datasets} datasets)",
                 fontsize=11, fontweight="bold")
    ax.grid(axis="x", alpha=0.3)
    utils.save_fig(fig, outpath, dpi=FIGURE_DPI)


def plot_grouped_rankings(scored, methods, group, outpath):
    """Rank `methods` among themselves over the axes of every
    RANKING_GROUPS level, in a single plot."""
    table = axis_scores(scored)
    sub_methods = [m for m in methods if m in table.index]
    present_groups = []
    for name, axes in RANKING_GROUPS.items():
        avail = [a for a in axes if a in table.columns
                 and table.loc[sub_methods, a].notna().any()]
        if avail:
            present_groups.append((name, avail))
    if not present_groups:
        return

    # Consistent hue order: the caller-grouped fixed order
    methods_sorted = [m for m in METHODS if m in sub_methods]
    hue_order = [utils.display_name(m) for m in methods_sorted]

    # Collect data for bars and points
    rows = []
    point_rows = []
    for name, group_axes in present_groups:
        sub_table = table.loc[sub_methods, group_axes]
        ranking = summary_ranking(scored, sub_methods, table=sub_table)
        for method in methods_sorted:
            if method not in ranking.index:
                continue
            score = ranking.loc[method, "Score"]
            method_name = utils.display_name(method)
            rows.append({
                "Group": name,
                "Model Name": method_name,
                "Score": score
            })
            # Axis scores for this method in this group
            method_scores = sub_table.loc[method, group_axes].dropna().values
            for s in method_scores:
                point_rows.append({
                    "Group": name,
                    "Model Name": method_name,
                    "Score": s
                })

    df_plot = pd.DataFrame(rows)
    df_points = pd.DataFrame(point_rows)
    group_names = [p[0] for p in present_groups]
    palette = {utils.display_name(m): utils.method_color(m) for m in methods_sorted}

    figwidth = max(8.4, len(group_names) * 1.6)
    # Fixed aspect ratio for consistent display height in the notebook
    ax = utils.bar_plot(
        df_plot, x="Group", y="Score", order=group_names,
        hue="Model Name", hue_order=hue_order, palette=palette,
        figsize=(figwidth, figwidth / 3.0), points=SUMMARY_POINTS, point_data=df_points,
        xlabel="", ylabel="Mean rank score (1 = best)",
        title=f"Grouped rankings of the {group} models",
        legend=True, legend_title="Model", rotation=0,
        hatch="joint", ylim=(0, 1.1)
    )

    ax.axhline(1.0, color="#cccccc", linewidth=0.8, zorder=1)
    ax.axhline(0.5, color="#999999", linewidth=0.8, linestyle="--", zorder=1)

    utils.save_fig(ax.get_figure(), outpath, dpi=FIGURE_DPI)


REFERENCE_LABEL = "Reference"
SEGMENTS_K_LABEL = "Segments (×10³)"


def _segment_count_label(model):
    """Legend label of a segment-count bar, "Reference" for the published
    markup."""
    return REFERENCE_LABEL if model == REFERENCE else utils.display_name(model)


def _segment_count_color(model):
    return utils.bin_color("reference") if model == REFERENCE \
        else utils.method_color(model)


PANEL_SIZE = (9.0, 4.8)

STATES_LABEL = "States"

SEGMENT_COUNT_VARIANTS = [("_all", [REFERENCE] + METHODS)]

# (column, file stem, title, y label, y scaling) per granularity figure.
GRANULARITY_FIGURES = [
    ("N_Segments", "segment_counts", "Number of segments per segmentation",
     SEGMENTS_K_LABEL, 1 / 1000.0),
    ("N_States", "state_counts", "Actual number of states per segmentation",
     STATES_LABEL, 1.0),
]

ANNOTATED_LABEL = "Genome outside the quiescent state (%)"

ANNOTATED_FIGURES = [
    ("Annotated", "annotated_share",
     "Share of the genome on a state other than the quiescent one",
     ANNOTATED_LABEL, 100.0),
]


def _segment_count_variant(counts, models):
    """`counts` cut to the segmentations a variant reads: the ENCODE joint
    models exist per replicate only, and the individual counts differ between
    the two scopes, so a figure with both families takes the replicate scope
    and keeps the pooled reference."""
    if not any(is_joint(m) for m in models):
        keep = counts["Scope"] != "replicates"
    else:
        keep = (counts["Scope"] != "pooled") | (counts["Model"] == REFERENCE)
    return counts[keep & counts["Model"].isin(models)]


def _per_model_figures(counts, figures, outdir):
    """One figure per (column of `figures`, model variant): the three datasets
    side by side, a bar per model, over the segmentations `counts` holds.

    Reference is the published markup ENCODE ships per dataset; SAGAconf has
    none, and the published 15-state joint model of the 1000 epigenomes is
    drawn as its joint ChromHMM instead."""
    written = []
    for column, stem, title, ylabel, scale in figures:
        for suffix, wanted in SEGMENT_COUNT_VARIANTS:
            rows = _segment_count_variant(counts, wanted).dropna(subset=[column])
            models = [m for m in wanted if (rows["Model"] == m).any()]
            if not models:
                continue
            datasets = [d for d in DATASETS if (rows["Dataset"] == d).any()]
            order = [DATASET_NAMES[d] for d in datasets]
            hue_order = [_segment_count_label(m) for m in models]
            frame = rows.assign(**{
                "Dataset Name": rows["Dataset"].astype(str).map(DATASET_NAMES),
                "Model Name": rows["Model"].astype(str).map(_segment_count_label),
                ylabel: rows[column] * scale})

            ax = utils.bar_plot(
                frame, "Dataset Name", ylabel, order=order,
                hue="Model Name", hue_order=hue_order,
                palette={_segment_count_label(m): _segment_count_color(m)
                         for m in models},
                figsize=PANEL_SIZE, rotation=0, hatch="joint",
                points={"size": 2.5, "alpha": 0.55, "jitter": 0.2},
                title=title, xlabel="Dataset", ylabel=ylabel, legend=True,
                legend_title="Model")
            note = None
            if (rows["Dataset"] == "encode").any():
                note = ("- ENCODE per-replicate segmentations, the scope its "
                        "joint models exist in; its reference markup is the "
                        "pooled one, it has no per-replicate counterpart"
                        if any(is_joint(m) for m in models)
                        else "- ENCODE pooled segmentations")
            name = f"{stem}{suffix}.png"
            utils.save_fig(ax.get_figure(), os.path.join(outdir, name),
                           dpi=FIGURE_DPI, note=note)
            written.append(name)
    return written


def plot_segment_counts(counts, outdir):
    """Segment and state counts of every segmentation, a bar per model.

    The state count is how many labels a segmentation actually uses, which is
    below the 15 states it was asked for wherever a model collapses some of
    them."""
    if counts.empty:
        print("  no segment counts to plot")
        return []
    return _per_model_figures(counts, GRANULARITY_FIGURES, outdir)


def plot_annotated_share(shares, outdir):
    """How much of the genome every segmentation puts on a state other than
    the quiescent one, a bar per model.

    The quiescent state is the one no mark reaches, so this is the share of
    the genome a model says something about at all. It is not a score on its
    own - a model can annotate more of the genome by spreading a weak state
    over it - but it is what the agreement metrics of the FULL domain are
    dominated by, and it is where the callers differ most."""
    if shares.empty:
        print("  no state coverage to plot")
        return []
    return _per_model_figures(shares, ANNOTATED_FIGURES, outdir)


def plot_annotated_share_chromhmm(shares, outdir):
    """Targeted comparison of the ChromHMM models and the reference on ENCODE
    and EPI1000."""
    if shares.empty:
        print("  no state coverage to plot")
        return []

    models = [REFERENCE, CHROMHMM_DEFAULT, JOINT_CHROMHMM]
    datasets = ["encode", "epi1000"]
    column, stem, title, ylabel, scale = ANNOTATED_FIGURES[0]

    rows = _segment_count_variant(shares, models).dropna(subset=[column])
    rows = rows[rows["Dataset"].isin(datasets)]
    # User request: don't show ChromHMM and ChromHMM joint for ENCODE, 
    # only show ENCODE reference, 15 state individual and 15 state joint models.
    rows = rows[((rows["Dataset"] == "encode") & (rows["Model"] == REFERENCE)) |
                ((rows["Dataset"] == "epi1000") & (rows["Model"].isin([CHROMHMM_DEFAULT, JOINT_CHROMHMM])))]
    if rows.empty:
        return []

    frame = rows.assign(**{
        "Legend Name": np.where((rows["Dataset"] == "encode") & (rows["Model"] == REFERENCE),
                                "ENCODE reference",
                                rows["Model"].map(_segment_count_label)),
        ylabel: rows[column] * scale})

    hue_order = ["ENCODE reference", _segment_count_label(CHROMHMM_DEFAULT),
                 _segment_count_label(JOINT_CHROMHMM)]
    hue_order = [h for h in hue_order if h in frame["Legend Name"].values]

    palette = {
        "ENCODE reference": "grey",
        _segment_count_label(CHROMHMM_DEFAULT): "lightcoral",
        _segment_count_label(JOINT_CHROMHMM): "skyblue"
    }

    ax = utils.bar_plot(
        frame, "Legend Name", ylabel, order=hue_order,
        hue="Legend Name", hue_order=hue_order,
        palette=palette,
        figsize=(4.0, 4.8), rotation=45, hatch="joint",
        points={"size": 2.5, "alpha": 0.55, "jitter": 0.2},
        title=title, xlabel="method", ylabel=ylabel, legend=False)

    name = f"{stem}_chromhmm.png"
    utils.save_fig(ax.get_figure(), os.path.join(outdir, name),
                   dpi=FIGURE_DPI)
    return [name]


SUMMARY_POINTS = {"size": 1.5, "alpha": 0.35, "jitter": 0.28}

POINTS_PER_BAR = 150

MODEL_VARIANTS = [("all", METHODS)]

COMPARISON_NOTES = {
    "cross_sample": "different samples segmented by the same model: every "
                    "same-assay dataset pair of ENCODE, every cell-line pair "
                    "of SAGAconf, a reproducible sample of 1000 epigenome "
                    "pairs",
    "replicates": "the two replicates of one biological condition, which is "
                  "the reproducibility ceiling of a model",
    "joint_indiv": "a sample segmented on its own against the same sample "
                   "inside a joint model, so each bar already compares the "
                   "two families and the figure has no joint variant",
}


def _model_variants(rows):
    """(suffix, models) per variant over the models `rows` holds, no
    repeats."""
    variants = []
    for suffix, models in MODEL_VARIANTS:
        present = tuple(m for m in models if (rows["Method"] == m).any())
        if not present or any(present == earlier for _, earlier in variants):
            continue
        variants.append((suffix, present))
    return variants


def _thin_points(frame, keys=("Dataset Name", "Model Name")):
    """At most POINTS_PER_BAR rows per bar with a fixed seed; None when nothing
    needs thinning."""
    groups = [group for _, group in frame.groupby(list(keys), observed=True)]
    if not any(len(group) > POINTS_PER_BAR for group in groups):
        return None
    return pd.concat([group if len(group) <= POINTS_PER_BAR
                      else group.sample(POINTS_PER_BAR, random_state=0)
                      for group in groups])


def _thinned_note(rows, keys):
    """The note of a figure whose densest bars are drawn thinned, or None."""
    sizes = rows.groupby(list(keys), observed=True).size()
    if sizes.empty or sizes.max() <= POINTS_PER_BAR:
        return None
    return (f"points thinned to {POINTS_PER_BAR} of the {int(sizes.max())} on "
            f"the densest bar; every bar is the mean over all of them")


def _model_bars(ax, rows, value_col, models, x_col, x_order, **kwargs):
    """One panel of a summary figure: a bar per model over the `x_col`
    levels."""
    frame = rows.assign(**{
        "Dataset Name": rows["Dataset"].astype(str).map(DATASET_NAMES),
        "Model Name": rows["Method"].astype(str).map(utils.display_name)})
    return utils.bar_plot(
        frame, x_col, value_col, ax=ax, order=list(x_order),
        hue="Model Name", hue_order=[utils.display_name(m) for m in models],
        palette={utils.display_name(m): utils.method_color(m) for m in models},
        points=SUMMARY_POINTS,
        point_data=_thin_points(frame, (x_col, "Model Name")),
        legend_title="Model", **kwargs)


def _summary_figure(rows, models, value_col, path, title, ylabel,
                    x_col="Dataset Name", x_order=None, panel_col=None,
                    panels=None, panel_labels=None, notes=(), ylim=None,
                    log=False, figwidth=PANEL_SIZE[0], rotation=0):
    """One summary figure: the `panels` levels of `panel_col` side by side, or
    a single panel."""
    if x_order is None:
        x_order = [DATASET_NAMES[d] for d in DATASETS
                   if (rows["Dataset"] == d).any()]
    keys = list(panels) if panels else [None]
    fig, axes = plt.subplots(1, len(keys),
                             figsize=(figwidth * len(keys), PANEL_SIZE[1]),
                             sharey=True, squeeze=False)
    for i, (ax, key) in enumerate(zip(axes[0], keys)):
        panel = rows if key is None else rows[rows[panel_col] == key]
        _model_bars(ax, panel, value_col, models, x_col, x_order,
                    title=None if key is None
                    else str((panel_labels or {}).get(key, key)),
                    xlabel="", ylabel=ylabel if i == 0 else "", ylim=ylim,
                    log=log, rotation=rotation, legend=(i == len(keys) - 1))
    fig.suptitle(title, fontsize=13, fontweight="bold")
    notes = [note for note in notes if note]
    utils.save_fig(fig, path, dpi=FIGURE_DPI,
                   note=f"- {'; '.join(notes)}" if notes else None)
    return os.path.basename(path)


def plot_comparison_figures(comparisons, outdir):
    """One figure per comparison, metric, domain and model variant."""
    written = []
    if comparisons.empty:
        print("  no comparison metrics to plot")
        return written
    for comparison in SUMMARY_COMPARISONS:
        for metric in SUMMARY_METRIC_ORDER:
            for domain in SUMMARY_DOMAINS:
                rows = comparisons[(comparisons["Comparison"] == comparison)
                                   & (comparisons["Metric"] == metric)
                                   & (comparisons["Domain"] == domain)]
                if rows.empty:
                    continue
                for suffix, models in _model_variants(rows):
                    name = (f"agreement_{comparison}_{utils.slug(metric)}"
                            f"_{domain}_{suffix}.png")
                    variant = rows[rows["Method"].isin(models)]
                    written.append(_summary_figure(
                        variant, models, "Value", os.path.join(outdir, name),
                        f"{axis_label(comparison, oneline=True)}: {metric} "
                        f"({utils.domain_display(domain)})",
                        ylabel=f"{metric} index", ylim=(0, 1.05),
                        notes=[_thinned_note(variant, ("Dataset", "Method"))]))
    return written


def plot_entropy_figures(entropy, outdir):
    """Transition matrix entropy, one figure per domain and model variant."""
    written = []
    if entropy.empty:
        print("  no entropy to plot")
        return written
    for domain in SUMMARY_DOMAINS:
        rows = entropy[entropy["Domain"] == domain]
        if rows.empty:
            continue
        missing = [DATASET_NAMES[d] for d in DATASETS
                   if (entropy["Dataset"] == d).any()
                   and not (rows["Dataset"] == d).any()]
        for suffix, models in _model_variants(rows):
            name = f"entropy_{domain}_{suffix}.png"
            variant = rows[rows["Method"].isin(models)]
            written.append(_summary_figure(
                variant, models, "Value", os.path.join(outdir, name),
                f"Transition matrix entropy ({utils.domain_display(domain)})",
                ylabel="Entropy (bits)",
                notes=[f"{', '.join(missing)} caches no model on this domain "
                       f"and is absent" if missing else None,
                       _thinned_note(variant, ("Dataset", "Method"))]))
    return written


def plot_peak_figures(peaks, outdir):
    """Peak count and mean peak length per mark, one panel per dataset."""
    written = []
    if peaks.empty:
        print("  no peak statistics to plot")
        return written
    models = [m for m in PEAK_MODELS if (peaks["Method"] == m).any()]
    datasets = [d for d in DATASETS if (peaks["Dataset"] == d).any()]
    marks = [m for m in PEAK_MARKS if (peaks["Mark"] == m).any()]
    by_mark = dict(x_col="Mark", x_order=marks, panel_col="Dataset",
                   panels=datasets, panel_labels=DATASET_NAMES, rotation=45,
                   figwidth=6.0)

    counts = peaks.assign(**{"Peaks (×10³)": peaks["N_Peaks"] / 1000.0})
    written.append(_summary_figure(
        counts, models, "Peaks (×10³)",
        os.path.join(outdir, "peaks_number_by_mark.png"),
        "Number of peaks per mark and model", ylabel="Peaks (×10³)",
        notes=[_thinned_note(counts, ("Dataset", "Method", "Mark"))],
        **by_mark))

    lengths = peaks[peaks["N_Peaks"] > 0]
    empty = len(peaks) - len(lengths)
    written.append(_summary_figure(
        lengths, models, "Mean_Length",
        os.path.join(outdir, "peaks_mean_length_by_mark.png"),
        "Mean peak length per mark and model",
        ylabel="Mean peak length (bp, log scale)", log=True,
        notes=[f"log scale; {empty} of the {len(peaks)} (sample, mark) calls "
               f"are empty, have no length and are left out here" if empty
               else "log scale",
               _thinned_note(lengths, ("Dataset", "Method", "Mark"))],
        **by_mark))
    return written


In [ ]:
plot_radar(table_scores, OUT)
plot_radar_values(table_values, OUT)
plot_radar_barplots(df_scored, table_scores, OUT)
plot_radar_value_barplots(df_scored, table_values, OUT)
for group, ranking, methods in [
        ("individual", df_ranking_individual, INDIVIDUAL_METHODS),
        ("joint", df_ranking_joint, JOINT_METHODS)]:
    plot_ranking(ranking, table_scores, group,
                 f"{OUT}/ranking_all_{group}.png")
    plot_grouped_rankings(df_scored, methods, group,
                          f"{OUT}/ranking_grouped_{group}.png")
SUMMARY_FIGURE_GROUPS = [
    ("Segmentation granularity",
     plot_segment_counts(df_segment_counts, OUT), 900),
    ("Genome outside the quiescent state",
     plot_annotated_share(df_annotated, OUT), 900),
    ("Transition matrix entropy",
     plot_entropy_figures(df_entropy_summary, OUT), 900),
    ("Agreement between segmentations",
     plot_comparison_figures(df_comparisons, OUT), 900),
]


# 3. Showing the results

In [ ]:
header("Feature coverage")
display(table_coverage)


In [ ]:
for group_title, figures, width in SUMMARY_FIGURE_GROUPS:
    header(group_title)
    show_all(figures, base=OUT, width=width)


In [ ]:
header("Rank profiles")
show_all(["radar_individual.png", "radar_joint.png",
          "radar_values_individual.png", "radar_values_joint.png",
          "radar_barplot_all.png", "radar_values_barplot_all.png"],
         base=OUT, width=900)

In [ ]:
for group, ranking in [("individual", df_ranking_individual),
                       ("joint", df_ranking_joint)]:
    header(f"{group.capitalize()} models summary ranks")
    display(ranking.round(3))
    show_all([f"ranking_all_{group}.png", f"ranking_grouped_{group}.png"],
             base=OUT, width=820)